In [ ]:
# W8.1.1 – Load + inspect + inner-join diagnostics (features + L2 labels)

from pathlib import Path
import pandas as pd
import numpy as np
import plotly.express as px

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 80)


FEATURES_PATH = Path("../data/derived/mru_usd_features_all_week5.csv")
L2_PATH       = Path("../data/derived/mru_usd_labels_L2_ma10_50_h5.csv")

print(f"FEATURES_PATH: {FEATURES_PATH.resolve()}")
print(f"FEATURES exists: {FEATURES_PATH.exists()}")
print(f"L2_PATH:       {L2_PATH.resolve()}")
print(f"L2 exists:     {L2_PATH.exists()}")

if not FEATURES_PATH.exists() or not L2_PATH.exists():
    missing = []
    if not FEATURES_PATH.exists():
        missing.append(str(FEATURES_PATH))
    if not L2_PATH.exists():
        missing.append(str(L2_PATH))
    print(f"Missing file(s): {missing}")
    raise SystemExit(1)

print("\n--- Load CSVs ---")
features_df = pd.read_csv(FEATURES_PATH)
l2_df = pd.read_csv(L2_PATH)

print("\n--- Basic shapes ---")
print(f"features_df: {features_df.shape}")
print(f"l2_df:       {l2_df.shape}")

print("\n--- Columns (features_df) ---")
print(list(features_df.columns))
print("\n--- Columns (l2_df) ---")
print(list(l2_df.columns))

print("\n--- Dtypes (features_df) ---")
print(features_df.dtypes)
print("\n--- Dtypes (l2_df) ---")
print(l2_df.dtypes)

print("\n--- Head(3) (features_df) ---")
print(features_df.head(3))
print("\n--- Head(3) (l2_df) ---")
print(l2_df.head(3))

if "date" not in features_df.columns or "date" not in l2_df.columns:
    print("\nNot provable from provided evidence.")
    raise SystemExit(1)

if "y_L2" not in l2_df.columns:
    print("\nNot provable from provided evidence.")
    raise SystemExit(1)

features_df["date"] = pd.to_datetime(features_df["date"], errors="coerce")
l2_df["date"] = pd.to_datetime(l2_df["date"], errors="coerce")

print(f"features_df date min/max: {features_df['date'].min()} → {features_df['date'].max()}")
print(f"l2_df date min/max:       {l2_df['date'].min()} → {l2_df['date'].max()}")

print(f"features_df NaNs in date: {features_df['date'].isna().sum()}")
print(f"l2_df NaNs in date:       {l2_df['date'].isna().sum()}")
print(f"l2_df NaNs in y_L2:       {l2_df['y_L2'].isna().sum()}")

print(f"features_df duplicated(date): {features_df['date'].duplicated().sum()}")
print(f"l2_df duplicated(date):       {l2_df['date'].duplicated().sum()}")

features_df = features_df.sort_values("date").copy()
l2_df = l2_df.sort_values("date").copy()

feat_dates = pd.Index(features_df["date"].dropna().unique())
lab_dates  = pd.Index(l2_df["date"].dropna().unique())

only_in_features = feat_dates.difference(lab_dates)
only_in_labels   = lab_dates.difference(feat_dates)
in_both          = feat_dates.intersection(lab_dates)

print(f"unique dates in features: {len(feat_dates)}")
print(f"unique dates in labels:   {len(lab_dates)}")
print(f"intersection (joinable):  {len(in_both)}")
print(f"only in features:         {len(only_in_features)}")
print(f"only in labels:           {len(only_in_labels)}")

merged = pd.merge(
    features_df,
    l2_df[["date", "y_L2"]],
    on="date",
    how="inner",
    validate="one_to_one" if (features_df["date"].is_unique and l2_df["date"].is_unique) else "many_to_many"
)

print(f"merged shape: {merged.shape}")
print(f"merged date min/max: {merged['date'].min()} → {merged['date'].max()}")

merged_sorted = merged.sort_values("date").reset_index(drop=True)
is_unique_dates = merged_sorted["date"].is_unique
is_monotonic = merged_sorted["date"].is_monotonic_increasing

print(f"Unique dates in merged: {is_unique_dates}")
print(f"Monotonic increasing dates: {is_monotonic}")


dist = merged_sorted["y_L2"].value_counts(dropna=False)
print(dist)

valid_labels = set(pd.Series(merged_sorted["y_L2"]).dropna().unique().tolist())
allowed = {-1, 0, 1}
subset_ok = valid_labels.issubset(allowed)

print(f"Observed non-NaN labels: {sorted(valid_labels)}")
print(f"Subset of {{-1,0,1}}: {subset_ok}")

merged_sorted["year"] = merged_sorted["date"].dt.year
yearly_counts = (
    merged_sorted
    .dropna(subset=["year"])
    .groupby(["year", "y_L2"])
    .size()
    .rename("count")
    .reset_index()
    .sort_values(["year", "y_L2"])
)

print(yearly_counts.to_string(index=False))

fig = px.scatter(
    merged_sorted,
    x="date",
    y="y_L2",
    title="W8.1 — L2 label (y_L2) over time (merged dataset)",
    labels={"date": "Date", "y_L2": "L2 label"},
)
fig.update_traces(marker=dict(size=3))
fig.show()

pass_one_table = merged_sorted.shape[0] > 0
pass_unique = bool(is_unique_dates)
pass_monotonic = bool(is_monotonic)
pass_labels = bool(subset_ok)

print(f"[{'PASS' if pass_one_table else 'FAIL'}] Non-empty merged table")
print(f"[{'PASS' if pass_unique else 'FAIL'}] No duplicated dates in merged")
print(f"[{'PASS' if pass_monotonic else 'FAIL'}] Strictly increasing date order after sort")
print(f"[{'PASS' if pass_labels else 'FAIL'}] y_L2 labels subset of {{-1,0,1}} (excluding NaN)")



=== Week 8 — W8.1.1 Load & Align Canonical Datasets ===

--- File existence checks (fixed relative paths) ---
FEATURES_PATH: C:\Users\simon\Desktop\my thesis\ThesisProject\data\derived\mru_usd_features_all_week5.csv
FEATURES exists: True
L2_PATH:       C:\Users\simon\Desktop\my thesis\ThesisProject\data\derived\mru_usd_labels_L2_ma10_50_h5.csv
L2 exists:     True

--- Load CSVs ---

--- Basic shapes ---
features_df: (6752, 15)
l2_df:       (6752, 6)

--- Columns (features_df) ---
['date', 'open', 'high', 'low', 'close', 'log_price', 'log_return', 'f_r_lag1', 'f_past_ret_5', 'f_past_ret_20', 'f_vol_10', 'f_vol_30', 'f_ma_50', 'f_dist_50', 'f_high_vol']

--- Columns (l2_df) ---
['date', 'close', 'MA_fast', 'MA_slow', 'MA_slow_slope', 'y_L2']

--- Dtypes (features_df) ---
date              object
open             float64
high             float64
low              float64
close            float64
log_price        float64
log_return       float64
f_r_lag1         float64
f_past_ret_5     flo


=== W8.1 Acceptance Gate ===
[PASS] Non-empty merged table
[PASS] No duplicated dates in merged
[PASS] Strictly increasing date order after sort
[PASS] y_L2 labels subset of {-1,0,1} (excluding NaN)

=== COPY OUTPUT BACK ===


In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 80)


FEATURES_PATH = Path("../data/derived/mru_usd_features_all_week5.csv")
L2_PATH       = Path("../data/derived/mru_usd_labels_L2_ma10_50_h5.csv")

if not FEATURES_PATH.exists() or not L2_PATH.exists():
    print("Not provable from provided evidence.")
    print(f"Exists features: {FEATURES_PATH.exists()} | Exists L2: {L2_PATH.exists()}")
    raise SystemExit(1)

features_df = pd.read_csv(FEATURES_PATH)
l2_df = pd.read_csv(L2_PATH)

if "date" not in features_df.columns or "date" not in l2_df.columns:
    raise SystemExit(1)

if "y_L2" not in l2_df.columns:
    print("Not provable from provided evidence.")

    raise SystemExit(1)

features_df["date"] = pd.to_datetime(features_df["date"], errors="coerce")
l2_df["date"] = pd.to_datetime(l2_df["date"], errors="coerce")

if features_df["date"].isna().any() or l2_df["date"].isna().any():
    print("Not provable from provided evidence.")

    raise SystemExit(1)

df = pd.merge(
    features_df.sort_values("date"),
    l2_df[["date", "y_L2"]].sort_values("date"),
    on="date",
    how="inner",
    validate="one_to_one"
).sort_values("date").reset_index(drop=True)

print(f"df shape: {df.shape}")
print(f"df date min/max: {df['date'].min()} → {df['date'].max()}")
print(f"df unique dates: {df['date'].is_unique} | df monotonic: {df['date'].is_monotonic_increasing}")

V1_TRAIN_START = pd.Timestamp("2000-01-03")
V1_TRAIN_END   = pd.Timestamp("2015-12-31")
V1_VAL_START   = pd.Timestamp("2016-01-01")
V1_VAL_END     = pd.Timestamp("2019-12-31")
V1_TEST_START  = pd.Timestamp("2020-01-01")
V1_TEST_END    = pd.Timestamp("2025-11-25")

v1_train = (df["date"] >= V1_TRAIN_START) & (df["date"] <= V1_TRAIN_END)
v1_val   = (df["date"] >= V1_VAL_START) & (df["date"] <= V1_VAL_END)
v1_test  = (df["date"] >= V1_TEST_START) & (df["date"] <= V1_TEST_END)

print("\n=== V1 diagnostics ===")
print(f"V1-train count: {v1_train.sum()} | range: {df.loc[v1_train,'date'].min()} → {df.loc[v1_train,'date'].max()}")
print(f"V1-val   count: {v1_val.sum()}   | range: {df.loc[v1_val,'date'].min()} → {df.loc[v1_val,'date'].max()}")
print(f"V1-test  count: {v1_test.sum()}  | range: {df.loc[v1_test,'date'].min()} → {df.loc[v1_test,'date'].max()}")

ov_train_val  = (v1_train & v1_val).sum()
ov_train_test = (v1_train & v1_test).sum()
ov_val_test   = (v1_val & v1_test).sum()

print("\n--- V1 overlap checks (must be 0) ---")
print(f"train ∩ val : {ov_train_val}")
print(f"train ∩ test: {ov_train_test}")
print(f"val   ∩ test: {ov_val_test}")

in_any_v1 = v1_train | v1_val | v1_test
outside_v1 = (~in_any_v1).sum()
print(f"\nDates outside V1 windows (should be 0 if df exactly matches modeling period): {outside_v1}")


folds = [
    ("F1", pd.Timestamp("2000-01-03"), pd.Timestamp("2009-12-31"), pd.Timestamp("2010-01-01"), pd.Timestamp("2011-12-31")),
    ("F2", pd.Timestamp("2000-01-03"), pd.Timestamp("2011-12-31"), pd.Timestamp("2012-01-01"), pd.Timestamp("2013-12-31")),
    ("F3", pd.Timestamp("2000-01-03"), pd.Timestamp("2013-12-31"), pd.Timestamp("2014-01-01"), pd.Timestamp("2015-12-31")),
    ("F4", pd.Timestamp("2000-01-03"), pd.Timestamp("2015-12-31"), pd.Timestamp("2016-01-01"), pd.Timestamp("2019-12-31")),
]

rows = []
print("\n=== V2 diagnostics (fold-by-fold) ===")
for fold_id, tr_s, tr_e, te_s, te_e in folds:
    tr_mask = (df["date"] >= tr_s) & (df["date"] <= tr_e)
    te_mask = (df["date"] >= te_s) & (df["date"] <= te_e)

    tr_n = int(tr_mask.sum())
    te_n = int(te_mask.sum())
    overlap = int((tr_mask & te_mask).sum())

    rows.append({
        "fold": fold_id,
        "train_start": tr_s.date(),
        "train_end": tr_e.date(),
        "train_n": tr_n,
        "test_start": te_s.date(),
        "test_end": te_e.date(),
        "test_n": te_n,
        "overlap_n": overlap,
        "train_end_before_test_start": (tr_e < te_s),
    })

v2_table = pd.DataFrame(rows)
print(v2_table.to_string(index=False))

# Extra V2 rule checks
print("\n--- V2 rule checks ---")
print(f"All folds train_end < test_start: {bool(v2_table['train_end_before_test_start'].all())}")
print(f"All folds have zero overlap_n:    {bool((v2_table['overlap_n'] == 0).all())}")

V2_GLOBAL_START = pd.Timestamp("2000-01-03")
V2_GLOBAL_END   = pd.Timestamp("2019-12-31")
v2_all_tests_mask = pd.Series(False, index=df.index)
for _, _, _, te_s, te_e in folds:
    v2_all_tests_mask |= (df["date"] >= te_s) & (df["date"] <= te_e)

v2_test_min = df.loc[v2_all_tests_mask, "date"].min()
v2_test_max = df.loc[v2_all_tests_mask, "date"].max()
print(f"V2 combined test min/max: {v2_test_min} → {v2_test_max}")
print(f"V2 test windows within [2000-01-03, 2019-12-31]: {bool((v2_test_min >= V2_GLOBAL_START) and (v2_test_max <= V2_GLOBAL_END))}")

pass_v1_overlaps = (ov_train_val == 0 and ov_train_test == 0 and ov_val_test == 0)
pass_v2 = bool(v2_table["train_end_before_test_start"].all() and (v2_table["overlap_n"] == 0).all())

print(f"[{'PASS' if pass_v1_overlaps else 'FAIL'}] V1 segments are non-overlapping")
print(f"[{'PASS' if pass_v2 else 'FAIL'}] V2 folds have train_end < test_start and zero overlap")

if not (pass_v1_overlaps and pass_v2):
    print("\nSTOP: Split-mask integrity failed.")
    print("Not provable from provided evidence.")
    raise SystemExit(1)



=== Week 8 — W8.2.1 Split masks (V1 + V2) + diagnostics ===

--- Joined dataset sanity ---
df shape: (6752, 16)
df date min/max: 2000-01-03 00:00:00 → 2025-11-25 00:00:00
df unique dates: True | df monotonic: True

=== V1 diagnostics ===
V1-train count: 4169 | range: 2000-01-03 00:00:00 → 2015-12-31 00:00:00
V1-val   count: 1043   | range: 2016-01-01 00:00:00 → 2019-12-31 00:00:00
V1-test  count: 1540  | range: 2020-01-01 00:00:00 → 2025-11-25 00:00:00

--- V1 overlap checks (must be 0) ---
train ∩ val : 0
train ∩ test: 0
val   ∩ test: 0

Dates outside V1 windows (should be 0 if df exactly matches modeling period): 0

=== V2 diagnostics (fold-by-fold) ===
fold train_start  train_end  train_n test_start   test_end  test_n  overlap_n  train_end_before_test_start
  F1  2000-01-03 2009-12-31     2604 2010-01-01 2011-12-31     521          0                         True
  F2  2000-01-03 2011-12-31     3125 2012-01-01 2013-12-31     522          0                         True
  F3  2000-01-0

In [ ]:
# W8.3.1 – E5 (L2, V1) DATA PREP 

from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 80)


FEATURES_PATH = Path("../data/derived/mru_usd_features_all_week5.csv")
L2_PATH       = Path("../data/derived/mru_usd_labels_L2_ma10_50_h5.csv")

if not FEATURES_PATH.exists() or not L2_PATH.exists():
    print("Not provable from provided evidence.")
    print(f"Exists features: {FEATURES_PATH.exists()} | Exists L2: {L2_PATH.exists()}")
    raise SystemExit(1)

# Reload (self-contained)
features_df = pd.read_csv(FEATURES_PATH)
l2_df = pd.read_csv(L2_PATH)

# Required columns (locked facts)
REQUIRED_JOIN_KEY = "date"
REQUIRED_LABEL_COL = "y_L2"
F_CORE = ["f_r_lag1", "f_past_ret_5", "f_past_ret_20", "f_vol_10", "f_vol_30"]

# Schema checks
missing = []
for col in [REQUIRED_JOIN_KEY]:
    if col not in features_df.columns: missing.append(f"features_df missing '{col}'")
    if col not in l2_df.columns:       missing.append(f"l2_df missing '{col}'")
if REQUIRED_LABEL_COL not in l2_df.columns:
    missing.append(f"l2_df missing '{REQUIRED_LABEL_COL}'")
for c in F_CORE:
    if c not in features_df.columns:
        missing.append(f"features_df missing F_CORE column '{c}'")

if missing:
    print("Schema mismatch(s):")
    for m in missing:
        print(" -", m)
    raise SystemExit(1)

features_df["date"] = pd.to_datetime(features_df["date"], errors="coerce")
l2_df["date"] = pd.to_datetime(l2_df["date"], errors="coerce")

if features_df["date"].isna().any() or l2_df["date"].isna().any():
    print("Not provable from provided evidence.")
    raise SystemExit(1)

df = pd.merge(
    features_df.sort_values("date"),
    l2_df[["date", "y_L2"]].sort_values("date"),
    on="date",
    how="inner",
    validate="one_to_one"
).sort_values("date").reset_index(drop=True)

print(f"df shape: {df.shape}")
print(f"df date min/max: {df['date'].min()} → {df['date'].max()}")
print(f"df unique dates: {df['date'].is_unique} | df monotonic: {df['date'].is_monotonic_increasing}")

V1_TRAIN_START = pd.Timestamp("2000-01-03")
V1_TRAIN_END   = pd.Timestamp("2015-12-31")
V1_VAL_START   = pd.Timestamp("2016-01-01")
V1_VAL_END     = pd.Timestamp("2019-12-31")
V1_TEST_START  = pd.Timestamp("2020-01-01")
V1_TEST_END    = pd.Timestamp("2025-11-25")

v1_train = (df["date"] >= V1_TRAIN_START) & (df["date"] <= V1_TRAIN_END)
v1_val   = (df["date"] >= V1_VAL_START) & (df["date"] <= V1_VAL_END)
v1_test  = (df["date"] >= V1_TEST_START) & (df["date"] <= V1_TEST_END)

print(f"train: {int(v1_train.sum())} | {df.loc[v1_train,'date'].min()} → {df.loc[v1_train,'date'].max()}")
print(f"val:   {int(v1_val.sum())}   | {df.loc[v1_val,'date'].min()} → {df.loc[v1_val,'date'].max()}")
print(f"test:  {int(v1_test.sum())}  | {df.loc[v1_test,'date'].min()} → {df.loc[v1_test,'date'].max()}")

nan_train = int(df.loc[v1_train, "y_L2"].isna().sum())
nan_val   = int(df.loc[v1_val,   "y_L2"].isna().sum())
nan_test  = int(df.loc[v1_test,  "y_L2"].isna().sum())

print("\n=== y_L2 NaN counts (by V1 segment) ===")
print(f"train y_L2 NaNs: {nan_train}")
print(f"val   y_L2 NaNs: {nan_val}")
print(f"test  y_L2 NaNs: {nan_test}")

print("\n=== y_L2 distribution (TRAIN, incl NaN) ===")
print(df.loc[v1_train, "y_L2"].value_counts(dropna=False).sort_index())
print("\n=== y_L2 distribution (VAL, incl NaN) ===")
print(df.loc[v1_val, "y_L2"].value_counts(dropna=False).sort_index())
print("\n=== y_L2 distribution (TEST, incl NaN) ===")
print(df.loc[v1_test, "y_L2"].value_counts(dropna=False).sort_index())

def nan_counts_segment(mask, name):
    sub = df.loc[mask, F_CORE]
    counts = sub.isna().sum().sort_values(ascending=False)
    top = counts[counts > 0]
    print(f"\n=== F_CORE NaN counts in {name} (only >0 shown) ===")
    if top.empty:
        print("(none)")
    else:
        print(top.to_string())

nan_counts_segment(v1_train, "TRAIN")
nan_counts_segment(v1_val,   "VAL")
nan_counts_segment(v1_test,  "TEST")

if (nan_train + nan_val + nan_test) > 0:
    print("\nNot provable from provided evidence.")
    print("y_L2 contains NaNs in at least one V1 segment (likely MA warmup).")
    raise SystemExit(0)


=== Week 8 — W8.3.1 E5 Data Prep (L2, V1) + NaN-policy gate ===

--- Joined dataset snapshot ---
df shape: (6752, 16)
df date min/max: 2000-01-03 00:00:00 → 2025-11-25 00:00:00
df unique dates: True | df monotonic: True

=== V1 segment counts ===
train: 4169 | 2000-01-03 00:00:00 → 2015-12-31 00:00:00
val:   1043   | 2016-01-01 00:00:00 → 2019-12-31 00:00:00
test:  1540  | 2020-01-01 00:00:00 → 2025-11-25 00:00:00

=== y_L2 NaN counts (by V1 segment) ===
train y_L2 NaNs: 54
val   y_L2 NaNs: 0
test  y_L2 NaNs: 0

=== y_L2 distribution (TRAIN, incl NaN) ===
y_L2
-1.0    1409
 0.0     767
 1.0    1939
 NaN      54
Name: count, dtype: int64

=== y_L2 distribution (VAL, incl NaN) ===
y_L2
-1.0    147
 0.0    188
 1.0    708
Name: count, dtype: int64

=== y_L2 distribution (TEST, incl NaN) ===
y_L2
-1.0    457
 0.0    268
 1.0    815
Name: count, dtype: int64

=== F_CORE NaN counts in TRAIN (only >0 shown) ===
f_vol_30         31
f_past_ret_20    20
f_vol_10         11
f_past_ret_5      5
f_

SystemExit: 0

c:\Users\simon\Desktop\my thesis\ThesisProject\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3707: UserWarning:

To exit: use 'exit', 'quit', or Ctrl-D.



In [ ]:
# V1: Extract clean X/y for L2 + F_CORE (DROP rows with any NaN in F_CORE ∪ {y_L2})

from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 80)


FEATURES_PATH = Path("../data/derived/mru_usd_features_all_week5.csv")
L2_PATH       = Path("../data/derived/mru_usd_labels_L2_ma10_50_h5.csv")

if not FEATURES_PATH.exists() or not L2_PATH.exists():
    print("Not provable from provided evidence.")
    print(f"Exists features: {FEATURES_PATH.exists()} | Exists L2: {L2_PATH.exists()}")
    raise RuntimeError("STOP: missing input files")

features_df = pd.read_csv(FEATURES_PATH)
l2_df = pd.read_csv(L2_PATH)

JOIN_KEY = "date"
LABEL_COL = "y_L2"
F_CORE = ["f_r_lag1", "f_past_ret_5", "f_past_ret_20", "f_vol_10", "f_vol_30"]

missing = []
for c in [JOIN_KEY]:
    if c not in features_df.columns: missing.append(f"features_df missing '{c}'")
    if c not in l2_df.columns:       missing.append(f"l2_df missing '{c}'")
if LABEL_COL not in l2_df.columns:
    missing.append(f"l2_df missing '{LABEL_COL}'")
for c in F_CORE:
    if c not in features_df.columns:
        missing.append(f"features_df missing F_CORE '{c}'")

if missing:
    print("Not provable from provided evidence.")
    print("Schema mismatch(s):")
    for m in missing:
        print(" -", m)
    raise RuntimeError("STOP: schema mismatch")

# Parse date
features_df["date"] = pd.to_datetime(features_df["date"], errors="coerce")
l2_df["date"] = pd.to_datetime(l2_df["date"], errors="coerce")

if features_df["date"].isna().any() or l2_df["date"].isna().any():
    print("Not provable from provided evidence.")
    raise RuntimeError("STOP: date parse failure")

df = pd.merge(
    features_df.sort_values("date"),
    l2_df[["date", "y_L2"]].sort_values("date"),
    on="date",
    how="inner",
    validate="one_to_one"
).sort_values("date").reset_index(drop=True)

print(f"df shape: {df.shape}")
print(f"df date min/max: {df['date'].min()} → {df['date'].max()}")
print(f"df unique dates: {df['date'].is_unique} | df monotonic: {df['date'].is_monotonic_increasing}")

# V1 boundaries (locked)
V1_TRAIN_START = pd.Timestamp("2000-01-03")
V1_TRAIN_END   = pd.Timestamp("2015-12-31")
V1_VAL_START   = pd.Timestamp("2016-01-01")
V1_VAL_END     = pd.Timestamp("2019-12-31")
V1_TEST_START  = pd.Timestamp("2020-01-01")
V1_TEST_END    = pd.Timestamp("2025-11-25")

m_train = (df["date"] >= V1_TRAIN_START) & (df["date"] <= V1_TRAIN_END)
m_val   = (df["date"] >= V1_VAL_START) & (df["date"] <= V1_VAL_END)
m_test  = (df["date"] >= V1_TEST_START) & (df["date"] <= V1_TEST_END)

print("\n=== V1 segment raw counts ===")
print(f"train: {int(m_train.sum())}")
print(f"val  : {int(m_val.sum())}")
print(f"test : {int(m_test.sum())}")

cols_to_check = F_CORE + [LABEL_COL]

def extract_segment(name, mask):
    subset = df.loc[mask, ["date"] + cols_to_check].copy()
    before = subset.shape[0]

    nan_counts = subset[cols_to_check].isna().sum().sort_values(ascending=False)
    any_nan_mask = subset[cols_to_check].isna().any(axis=1)
    dropped = int(any_nan_mask.sum())

    subset_clean = subset.loc[~any_nan_mask].copy()
    after = subset_clean.shape[0]

    print(f"\n--- {name}: NaN diagnostics over selected columns (F_CORE ∪ {{y_L2}}) ---")
    print(f"Rows before dropna: {before}")
    print(f"Rows dropped (any NaN): {dropped}")
    print(f"Rows after dropna: {after}")

    top = nan_counts[nan_counts > 0]
    print("\nNaN counts per column (only >0 shown):")
    if top.empty:
        print("(none)")
    else:
        print(top.to_string())

    if after > 0:
        print(f"\n{ name } date min/max after dropna: {subset_clean['date'].min()} → {subset_clean['date'].max()}")
        print(f"{ name } y_L2 distribution after dropna:")
        print(subset_clean["y_L2"].value_counts(dropna=False).sort_index())
    else:
        print("Not provable from provided evidence.")
        print(f"After dropna, {name} has 0 rows.")
        raise RuntimeError("STOP: empty segment after dropna")

    X = subset_clean[F_CORE].to_numpy()
    y = subset_clean["y_L2"].to_numpy()
    df_keep = subset_clean[["date"]].copy()
    return X, y, df_keep

X_train_L2, y_train_L2, df_train_L2 = extract_segment("TRAIN", m_train)
X_val_L2,   y_val_L2,   df_val_L2   = extract_segment("VAL",   m_val)
X_test_L2,  y_test_L2,  df_test_L2  = extract_segment("TEST",  m_test)

print("\n=== Summary of V1 shapes for L2 (after NaN filtering) ===")
print(f"train: X={X_train_L2.shape}, y={y_train_L2.shape}, date min/max = {df_train_L2['date'].min()} → {df_train_L2['date'].max()}")
print(f"val  : X={X_val_L2.shape}, y={y_val_L2.shape}, date min/max = {df_val_L2['date'].min()} → {df_val_L2['date'].max()}")
print(f"test : X={X_test_L2.shape}, y={y_test_L2.shape}, date min/max = {df_test_L2['date'].min()} → {df_test_L2['date'].max()}")



=== Week 8 — W8.3.2 V1 extraction diagnostics (L2 + F_CORE, DROP any NaN) ===

--- Joined dataset snapshot ---
df shape: (6752, 16)
df date min/max: 2000-01-03 00:00:00 → 2025-11-25 00:00:00
df unique dates: True | df monotonic: True

=== V1 segment raw counts ===
train: 4169
val  : 1043
test : 1540

--- TRAIN: NaN diagnostics over selected columns (F_CORE ∪ {y_L2}) ---
Rows before dropna: 4169
Rows dropped (any NaN): 54
Rows after dropna: 4115

NaN counts per column (only >0 shown):
y_L2             54
f_vol_30         31
f_past_ret_20    20
f_vol_10         11
f_past_ret_5      5
f_r_lag1          2

TRAIN date min/max after dropna: 2000-03-17 00:00:00 → 2015-12-31 00:00:00
TRAIN y_L2 distribution after dropna:
y_L2
-1.0    1409
 0.0     767
 1.0    1939
Name: count, dtype: int64

--- VAL: NaN diagnostics over selected columns (F_CORE ∪ {y_L2}) ---
Rows before dropna: 1043
Rows dropped (any NaN): 0
Rows after dropna: 1043

NaN counts per column (only >0 shown):
(none)

VAL date min/m

In [ ]:
# W8.3.3 – E5 (LogReg, L2, V1): train-only scaling + fit + evaluation + diagnostics 

from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

import plotly.express as px

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 80)


FEATURES_PATH = Path("../data/derived/mru_usd_features_all_week5.csv")
L2_PATH       = Path("../data/derived/mru_usd_labels_L2_ma10_50_h5.csv")

METRICS_OUT = Path("../results/tables/metrics_E5.csv")
PRED_DIR = Path("../results/predictions")
PRED_OUT = PRED_DIR / "predictions_E5_V1_test.csv"

if not FEATURES_PATH.exists() or not L2_PATH.exists():
    print("Not provable from provided evidence.")
    print(f"Exists features: {FEATURES_PATH.exists()} | Exists L2: {L2_PATH.exists()}")
    raise RuntimeError("STOP: missing input files")

if not METRICS_OUT.parent.exists():
    print("Not provable from provided evidence.")
    print(f"Missing metrics output directory: {METRICS_OUT.parent}")
    raise RuntimeError("STOP: missing metrics directory")

PRED_DIR.mkdir(parents=True, exist_ok=True)

features_df = pd.read_csv(FEATURES_PATH)
l2_df = pd.read_csv(L2_PATH)

JOIN_KEY = "date"
LABEL_COL = "y_L2"
F_CORE = ["f_r_lag1", "f_past_ret_5", "f_past_ret_20", "f_vol_10", "f_vol_30"]

for col in [JOIN_KEY]:
    if col not in features_df.columns or col not in l2_df.columns:
        print("Not provable from provided evidence.")
        raise RuntimeError("STOP: schema mismatch")

if LABEL_COL not in l2_df.columns:
    print("Not provable from provided evidence.")
    raise RuntimeError("STOP: schema mismatch")

for c in F_CORE:
    if c not in features_df.columns:
        print("Not provable from provided evidence.")
        raise RuntimeError("STOP: schema mismatch")

features_df["date"] = pd.to_datetime(features_df["date"], errors="coerce")
l2_df["date"] = pd.to_datetime(l2_df["date"], errors="coerce")

if features_df["date"].isna().any() or l2_df["date"].isna().any():
    print("Not provable from provided evidence.")
    raise RuntimeError("STOP: date parse failure")

# Join (one-to-one)
df = pd.merge(
    features_df.sort_values("date"),
    l2_df[["date", "y_L2"]].sort_values("date"),
    on="date",
    how="inner",
    validate="one_to_one"
).sort_values("date").reset_index(drop=True)

# V1 boundaries (locked)
V1_TRAIN_START = pd.Timestamp("2000-01-03")
V1_TRAIN_END   = pd.Timestamp("2015-12-31")
V1_VAL_START   = pd.Timestamp("2016-01-01")
V1_VAL_END     = pd.Timestamp("2019-12-31")
V1_TEST_START  = pd.Timestamp("2020-01-01")
V1_TEST_END    = pd.Timestamp("2025-11-25")

m_train = (df["date"] >= V1_TRAIN_START) & (df["date"] <= V1_TRAIN_END)
m_val   = (df["date"] >= V1_VAL_START) & (df["date"] <= V1_VAL_END)
m_test  = (df["date"] >= V1_TEST_START) & (df["date"] <= V1_TEST_END)

cols_to_check = F_CORE + [LABEL_COL]

def extract_clean(mask, name):
    subset = df.loc[mask, ["date"] + cols_to_check].copy()
    before = subset.shape[0]
    any_nan = subset[cols_to_check].isna().any(axis=1)
    dropped = int(any_nan.sum())
    subset = subset.loc[~any_nan].copy()
    after = subset.shape[0]

    print(f"\n--- {name} extraction (DROP any NaN in F_CORE ∪ {{y_L2}}) ---")
    print(f"rows before: {before} | dropped: {dropped} | after: {after}")
    if after == 0:
        print("Not provable from provided evidence.")
        print(f"{name} became empty after NaN filtering.")
        raise RuntimeError("STOP: empty segment")

    X = subset[F_CORE].to_numpy()
    y = subset["y_L2"].to_numpy()
    dates = subset["date"].copy()
    return X, y, dates

X_train, y_train, d_train = extract_clean(m_train, "TRAIN")
X_val,   y_val,   d_val   = extract_clean(m_val,   "VAL")
X_test,  y_test,  d_test  = extract_clean(m_test,  "TEST")

def cast_labels(y, name):
    y_ser = pd.Series(y)
    if y_ser.isna().any():
        print("Not provable from provided evidence.")
        print(f"{name} contains NaN labels after dropna; this should not happen.")
        raise RuntimeError("STOP: NaN labels remain")

    # check integral floats
    if np.issubdtype(y_ser.dtype, np.floating):
        if np.all(np.isclose(y_ser.values, np.round(y_ser.values))):
            return y_ser.round(0).astype(int).values
        else:
            print("Not provable from provided evidence.")
            print(f"{name} labels are non-integer floats; cannot safely cast.")
            raise RuntimeError("STOP: invalid label dtype")
    return y_ser.astype(int).values if np.issubdtype(y_ser.dtype, np.integer) else y_ser.values

y_train_i = cast_labels(y_train, "TRAIN")
y_val_i   = cast_labels(y_val,   "VAL")
y_test_i  = cast_labels(y_test,  "TEST")

print("TRAIN:", pd.Series(y_train_i).value_counts().sort_index().to_dict())
print("VAL  :", pd.Series(y_val_i).value_counts().sort_index().to_dict())
print("TEST :", pd.Series(y_test_i).value_counts().sort_index().to_dict())

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s   = scaler.transform(X_val)
X_test_s  = scaler.transform(X_test)

print("Scaler fit on TRAIN only: OK")
print(f"X_train_s mean (first 5 features): {np.mean(X_train_s, axis=0)[:5]}")
print(f"X_train_s std  (first 5 features): {np.std(X_train_s, axis=0)[:5]}")

model = LogisticRegression(
    C=1.0,
    penalty="l2",
    solver="lbfgs",
    multi_class="auto",
    max_iter=1000,
    class_weight=None,
)

model.fit(X_train_s, y_train_i)

classes = model.classes_
print(classes)

# Predict
pred_train = model.predict(X_train_s)
pred_val   = model.predict(X_val_s)
pred_test  = model.predict(X_test_s)

# Metrics
def metrics_block(y_true, y_pred, split_name):
    acc = accuracy_score(y_true, y_pred)
    f1m = f1_score(y_true, y_pred, average="macro")
    f1p = f1_score(y_true, y_pred, average=None, labels=classes)
    out = {
        "split": split_name,
        "accuracy": acc,
        "macro_f1": f1m,
    }
    # per-class f1 in classes_ order
    for c, v in zip(classes, f1p):
        out[f"f1_class_{c}"] = v
    return out

rows = []
rows.append(metrics_block(y_train_i, pred_train, "V1_train"))
rows.append(metrics_block(y_val_i,   pred_val,   "V1_val"))
rows.append(metrics_block(y_test_i,  pred_test,  "V1_test"))

metrics_df = pd.DataFrame(rows)
print(metrics_df.to_string(index=False))

# Confusion matrix (TEST)
cm = confusion_matrix(y_test_i, pred_test, labels=classes)
cm_df = pd.DataFrame(cm, index=[f"true_{c}" for c in classes], columns=[f"pred_{c}" for c in classes])

print(cm_df.to_string())

cm_long = cm_df.reset_index().melt(id_vars="index", var_name="pred", value_name="count")
fig_cm = px.imshow(
    cm,
    x=[str(c) for c in classes],
    y=[str(c) for c in classes],
    text_auto=True,
    title="E5 (LogReg, L2, V1) — Confusion Matrix on V1_test (labels=model.classes_)",
    labels=dict(x="Predicted", y="True", color="Count"),
)
fig_cm.show()

metrics_df.to_csv(METRICS_OUT, index=False)
print(f"\n[WRITE] metrics saved: {METRICS_OUT}")

pred_out_df = pd.DataFrame({
    "date": d_test.values,
    "y_true": y_test_i,
    "y_pred": pred_test,
})
pred_out_df.to_csv(PRED_OUT, index=False)
print(f"[WRITE] predictions saved: {PRED_OUT}")



=== Week 8 — W8.3.3 E5 (LogReg, L2, V1) train/eval diagnostics ===

--- TRAIN extraction (DROP any NaN in F_CORE ∪ {y_L2}) ---
rows before: 4169 | dropped: 54 | after: 4115

--- VAL extraction (DROP any NaN in F_CORE ∪ {y_L2}) ---
rows before: 1043 | dropped: 0 | after: 1043

--- TEST extraction (DROP any NaN in F_CORE ∪ {y_L2}) ---
rows before: 1540 | dropped: 0 | after: 1540

--- Label distribution (TRAIN/VAL/TEST) after filtering ---
TRAIN: {-1: 1409, 0: 767, 1: 1939}
VAL  : {-1: 147, 0: 188, 1: 708}
TEST : {-1: 457, 0: 268, 1: 815}

--- Scaling diagnostics ---
Scaler fit on TRAIN only: OK
X_train_s mean (first 5 features): [7.77021218e-18 1.72671382e-18 0.00000000e+00 2.41739934e-17
 5.52548421e-17]
X_train_s std  (first 5 features): [1. 1. 1. 1. 1.]

--- Model classes_ (used for label order in confusion matrices) ---
[-1  0  1]

=== TEXT MIRROR: E5 metrics table (classes_ order for per-class F1) ===
   split  accuracy  macro_f1  f1_class_-1  f1_class_0  f1_class_1
V1_train  0.6252

c:\Users\simon\Desktop\my thesis\ThesisProject\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning:

'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.




[WRITE] metrics saved: ..\results\tables\metrics_E5.csv
[WRITE] predictions saved: ..\results\predictions\predictions_E5_V1_test.csv

=== COPY OUTPUT BACK ===


In [ ]:
# W8.4.1 – E6 (LinearSVC, L2, V1): train-only scaling + fit + evaluation + diagnostics + save metrics/predictions

from pathlib import Path
import pandas as pd
import numpy as np

import sklearn
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

import plotly.express as px

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 80)



# Fixed notebook-relative paths
FEATURES_PATH = Path("../data/derived/mru_usd_features_all_week5.csv")
L2_PATH       = Path("../data/derived/mru_usd_labels_L2_ma10_50_h5.csv")

# Output paths (deterministic)
METRICS_OUT = Path("../results/tables/metrics_E6.csv")
PRED_DIR = Path("../results/predictions")
PRED_OUT = PRED_DIR / "predictions_E6_V1_test.csv"

if not FEATURES_PATH.exists() or not L2_PATH.exists():
    print("Not provable from provided evidence.")
    print(f"Exists features: {FEATURES_PATH.exists()} | Exists L2: {L2_PATH.exists()}")
    raise RuntimeError("STOP: missing input files")

if not METRICS_OUT.parent.exists():
    print("Not provable from provided evidence.")
    print(f"Missing metrics output directory: {METRICS_OUT.parent}")
    raise RuntimeError("STOP: missing metrics directory")

PRED_DIR.mkdir(parents=True, exist_ok=True)

# Reload (self-contained)
features_df = pd.read_csv(FEATURES_PATH)
l2_df = pd.read_csv(L2_PATH)

JOIN_KEY = "date"
LABEL_COL = "y_L2"
F_CORE = ["f_r_lag1", "f_past_ret_5", "f_past_ret_20", "f_vol_10", "f_vol_30"]

# Schema checks
missing = []
if JOIN_KEY not in features_df.columns: missing.append("features_df missing 'date'")
if JOIN_KEY not in l2_df.columns:       missing.append("l2_df missing 'date'")
if LABEL_COL not in l2_df.columns:      missing.append("l2_df missing 'y_L2'")
for c in F_CORE:
    if c not in features_df.columns:
        missing.append(f"features_df missing F_CORE '{c}'")

if missing:
    print("Not provable from provided evidence.")
    print("Schema mismatch(s):")
    for m in missing:
        print(" -", m)
    raise RuntimeError("STOP: schema mismatch")

# Parse date
features_df["date"] = pd.to_datetime(features_df["date"], errors="coerce")
l2_df["date"] = pd.to_datetime(l2_df["date"], errors="coerce")

if features_df["date"].isna().any() or l2_df["date"].isna().any():
    print("Not provable from provided evidence.")
    raise RuntimeError("STOP: date parse failure")

# Join (one-to-one)
df = pd.merge(
    features_df.sort_values("date"),
    l2_df[["date", "y_L2"]].sort_values("date"),
    on="date",
    how="inner",
    validate="one_to_one"
).sort_values("date").reset_index(drop=True)

# V1 boundaries (locked)
V1_TRAIN_START = pd.Timestamp("2000-01-03")
V1_TRAIN_END   = pd.Timestamp("2015-12-31")
V1_VAL_START   = pd.Timestamp("2016-01-01")
V1_VAL_END     = pd.Timestamp("2019-12-31")
V1_TEST_START  = pd.Timestamp("2020-01-01")
V1_TEST_END    = pd.Timestamp("2025-11-25")

m_train = (df["date"] >= V1_TRAIN_START) & (df["date"] <= V1_TRAIN_END)
m_val   = (df["date"] >= V1_VAL_START) & (df["date"] <= V1_VAL_END)
m_test  = (df["date"] >= V1_TEST_START) & (df["date"] <= V1_TEST_END)

cols_to_check = F_CORE + [LABEL_COL]

def extract_clean(mask, name):
    subset = df.loc[mask, ["date"] + cols_to_check].copy()
    before = subset.shape[0]
    any_nan = subset[cols_to_check].isna().any(axis=1)
    dropped = int(any_nan.sum())
    subset = subset.loc[~any_nan].copy()
    after = subset.shape[0]

    print(f"\n--- {name} extraction (DROP any NaN in F_CORE ∪ {{y_L2}}) ---")
    print(f"rows before: {before} | dropped: {dropped} | after: {after}")
    if after == 0:
        print("Not provable from provided evidence.")
        print(f"{name} became empty after NaN filtering.")
        raise RuntimeError("STOP: empty segment")

    X = subset[F_CORE].to_numpy()
    y = subset["y_L2"].to_numpy()
    dates = subset["date"].copy()
    return X, y, dates

X_train, y_train, d_train = extract_clean(m_train, "TRAIN")
X_val,   y_val,   d_val   = extract_clean(m_val,   "VAL")
X_test,  y_test,  d_test  = extract_clean(m_test,  "TEST")

def cast_labels(y, name):
    y_ser = pd.Series(y)
    if y_ser.isna().any():
        print("Not provable from provided evidence.")
        print(f"{name} contains NaN labels after dropna; this should not happen.")
        raise RuntimeError("STOP: NaN labels remain")

    if np.issubdtype(y_ser.dtype, np.floating):
        if np.all(np.isclose(y_ser.values, np.round(y_ser.values))):
            return y_ser.round(0).astype(int).values
        print("Not provable from provided evidence.")
        print(f"{name} labels are non-integer floats; cannot safely cast.")
        raise RuntimeError("STOP: invalid label dtype")
    return y_ser.astype(int).values if np.issubdtype(y_ser.dtype, np.integer) else y_ser.values

y_train_i = cast_labels(y_train, "TRAIN")
y_val_i   = cast_labels(y_val,   "VAL")
y_test_i  = cast_labels(y_test,  "TEST")

print("TRAIN:", pd.Series(y_train_i).value_counts().sort_index().to_dict())
print("VAL  :", pd.Series(y_val_i).value_counts().sort_index().to_dict())
print("TEST :", pd.Series(y_test_i).value_counts().sort_index().to_dict())

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s   = scaler.transform(X_val)
X_test_s  = scaler.transform(X_test)

print(f"X_train_s mean (first 5 features): {np.mean(X_train_s, axis=0)[:5]}")
print(f"X_train_s std  (first 5 features): {np.std(X_train_s, axis=0)[:5]}")

model = LinearSVC(
    C=1.0,
    class_weight=None,
    max_iter=5000,
)

model.fit(X_train_s, y_train_i)

classes = model.classes_
print(classes)

pred_train = model.predict(X_train_s)
pred_val   = model.predict(X_val_s)
pred_test  = model.predict(X_test_s)

def metrics_block(y_true, y_pred, split_name):
    acc = accuracy_score(y_true, y_pred)
    f1m = f1_score(y_true, y_pred, average="macro")
    f1p = f1_score(y_true, y_pred, average=None, labels=classes)
    out = {"split": split_name, "accuracy": acc, "macro_f1": f1m}
    for c, v in zip(classes, f1p):
        out[f"f1_class_{c}"] = v
    return out

metrics_df = pd.DataFrame([
    metrics_block(y_train_i, pred_train, "V1_train"),
    metrics_block(y_val_i,   pred_val,   "V1_val"),
    metrics_block(y_test_i,  pred_test,  "V1_test"),
])

print(metrics_df.to_string(index=False))

cm = confusion_matrix(y_test_i, pred_test, labels=classes)
cm_df = pd.DataFrame(cm, index=[f"true_{c}" for c in classes], columns=[f"pred_{c}" for c in classes])

print(cm_df.to_string())

fig_cm = px.imshow(
    cm,
    x=[str(c) for c in classes],
    y=[str(c) for c in classes],
    text_auto=True,
    title="E6 (LinearSVC, L2, V1) — Confusion Matrix on V1_test (labels=model.classes_)",
    labels=dict(x="Predicted", y="True", color="Count"),
)
fig_cm.show()

# Save artifacts
metrics_df.to_csv(METRICS_OUT, index=False)
print(f"\n[WRITE] metrics saved: {METRICS_OUT}")

pred_out_df = pd.DataFrame({"date": d_test.values, "y_true": y_test_i, "y_pred": pred_test})
pred_out_df.to_csv(PRED_OUT, index=False)
print(f"[WRITE] predictions saved: {PRED_OUT}")




=== Week 8 — W8.4.1 E6 (LinearSVC, L2, V1) train/eval diagnostics ===

--- Package versions (record for determinism) ---
sklearn: 1.7.2
pandas : 2.3.3
numpy  : 2.3.5

--- TRAIN extraction (DROP any NaN in F_CORE ∪ {y_L2}) ---
rows before: 4169 | dropped: 54 | after: 4115

--- VAL extraction (DROP any NaN in F_CORE ∪ {y_L2}) ---
rows before: 1043 | dropped: 0 | after: 1043

--- TEST extraction (DROP any NaN in F_CORE ∪ {y_L2}) ---
rows before: 1540 | dropped: 0 | after: 1540

--- Label distribution (TRAIN/VAL/TEST) after filtering ---
TRAIN: {-1: 1409, 0: 767, 1: 1939}
VAL  : {-1: 147, 0: 188, 1: 708}
TEST : {-1: 457, 0: 268, 1: 815}

--- Scaling diagnostics ---
Scaler fit on TRAIN only: OK
X_train_s mean (first 5 features): [7.77021218e-18 1.72671382e-18 0.00000000e+00 2.41739934e-17
 5.52548421e-17]
X_train_s std  (first 5 features): [1. 1. 1. 1. 1.]

--- Model classes_ (used for label order in confusion matrices) ---
[-1  0  1]

=== TEXT MIRROR: E6 metrics table (classes_ order for p


[WRITE] metrics saved: ..\results\tables\metrics_E6.csv
[WRITE] predictions saved: ..\results\predictions\predictions_E6_V1_test.csv

=== COPY OUTPUT BACK ===


In [ ]:
# W8.5.1 – E7 (LogReg, L2, V2): fold-by-fold scaling/train/eval + save metrics + save predictions per fold

from pathlib import Path
import pandas as pd
import numpy as np

import sklearn
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

import plotly.express as px

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 100)



FEATURES_PATH = Path("../data/derived/mru_usd_features_all_week5.csv")
L2_PATH       = Path("../data/derived/mru_usd_labels_L2_ma10_50_h5.csv")

# Output paths (deterministic)
METRICS_OUT = Path("../results/tables/metrics_E7.csv")
PRED_DIR = Path("../results/predictions")

if not FEATURES_PATH.exists() or not L2_PATH.exists():
    print("Not provable from provided evidence.")
    print(f"Exists features: {FEATURES_PATH.exists()} | Exists L2: {L2_PATH.exists()}")
    raise RuntimeError("STOP: missing input files")

if not METRICS_OUT.parent.exists():
    print("Not provable from provided evidence.")
    print(f"Missing metrics output directory: {METRICS_OUT.parent}")
    raise RuntimeError("STOP: missing metrics directory")

PRED_DIR.mkdir(parents=True, exist_ok=True)

features_df = pd.read_csv(FEATURES_PATH)
l2_df = pd.read_csv(L2_PATH)

JOIN_KEY = "date"
LABEL_COL = "y_L2"
F_CORE = ["f_r_lag1", "f_past_ret_5", "f_past_ret_20", "f_vol_10", "f_vol_30"]

# Schema checks
missing = []
if JOIN_KEY not in features_df.columns: missing.append("features_df missing 'date'")
if JOIN_KEY not in l2_df.columns:       missing.append("l2_df missing 'date'")
if LABEL_COL not in l2_df.columns:      missing.append("l2_df missing 'y_L2'")
for c in F_CORE:
    if c not in features_df.columns:
        missing.append(f"features_df missing F_CORE '{c}'")

if missing:
    print("Not provable from provided evidence.")
    print("Schema mismatch(s):")
    for m in missing:
        print(" -", m)
    raise RuntimeError("STOP: schema mismatch")

# Parse date
features_df["date"] = pd.to_datetime(features_df["date"], errors="coerce")
l2_df["date"] = pd.to_datetime(l2_df["date"], errors="coerce")

if features_df["date"].isna().any() or l2_df["date"].isna().any():
    print("Not provable from provided evidence.")
    raise RuntimeError("STOP: date parse failure")

# Join (one-to-one)
df = pd.merge(
    features_df.sort_values("date"),
    l2_df[["date", "y_L2"]].sort_values("date"),
    on="date",
    how="inner",
    validate="one_to_one"
).sort_values("date").reset_index(drop=True)

print(f"df shape: {df.shape}")
print(f"df date min/max: {df['date'].min()} → {df['date'].max()}")
print(f"df unique dates: {df['date'].is_unique} | df monotonic: {df['date'].is_monotonic_increasing}")

# V2 folds (locked; from your PASS output)
folds = [
    ("F1", pd.Timestamp("2000-01-03"), pd.Timestamp("2009-12-31"), pd.Timestamp("2010-01-01"), pd.Timestamp("2011-12-31")),
    ("F2", pd.Timestamp("2000-01-03"), pd.Timestamp("2011-12-31"), pd.Timestamp("2012-01-01"), pd.Timestamp("2013-12-31")),
    ("F3", pd.Timestamp("2000-01-03"), pd.Timestamp("2013-12-31"), pd.Timestamp("2014-01-01"), pd.Timestamp("2015-12-31")),
    ("F4", pd.Timestamp("2000-01-03"), pd.Timestamp("2015-12-31"), pd.Timestamp("2016-01-01"), pd.Timestamp("2019-12-31")),
]

cols_to_check = F_CORE + [LABEL_COL]

def cast_labels(y, name):
    y_ser = pd.Series(y)
    if y_ser.isna().any():
        print("Not provable from provided evidence.")
        print(f"{name} contains NaN labels after dropna; this should not happen.")
        raise RuntimeError("STOP: NaN labels remain")

    if np.issubdtype(y_ser.dtype, np.floating):
        if np.all(np.isclose(y_ser.values, np.round(y_ser.values))):
            return y_ser.round(0).astype(int).values
        print("Not provable from provided evidence.")
        print(f"{name} labels are non-integer floats; cannot safely cast.")
        raise RuntimeError("STOP: invalid label dtype")
    return y_ser.astype(int).values if np.issubdtype(y_ser.dtype, np.integer) else y_ser.values

def extract_clean_window(mask, name):
    subset = df.loc[mask, ["date"] + cols_to_check].copy()
    before = subset.shape[0]
    any_nan = subset[cols_to_check].isna().any(axis=1)
    dropped = int(any_nan.sum())
    subset = subset.loc[~any_nan].copy()
    after = subset.shape[0]

    print(f"\n--- {name} extraction (DROP any NaN in F_CORE ∪ {{y_L2}}) ---")
    print(f"rows before: {before} | dropped: {dropped} | after: {after}")
    if after == 0:
        print("Not provable from provided evidence.")
        print(f"{name} became empty after NaN filtering.")
        raise RuntimeError("STOP: empty window after dropna")

    X = subset[F_CORE].to_numpy()
    y = cast_labels(subset["y_L2"].to_numpy(), name)
    dates = subset["date"].copy()
    return X, y, dates

# Model constructor (pinned from Week 6 V2)
model_kwargs = dict(
    C=1.0,
    penalty="l2",
    solver="lbfgs",
    multi_class="multinomial",
    max_iter=200,
    class_weight=None,
)

print("\n--- E7 model constructor (pinned) ---")
print(model_kwargs)

metrics_rows = []
all_fold_metrics = []

for fold_id, tr_s, tr_e, te_s, te_e in folds:
    print("\n" + "="*80)
    print(f"=== Fold {fold_id} ===")
    print(f"Train window: {tr_s.date()} → {tr_e.date()}")
    print(f"Test  window: {te_s.date()} → {te_e.date()}")

    m_train = (df["date"] >= tr_s) & (df["date"] <= tr_e)
    m_test  = (df["date"] >= te_s) & (df["date"] <= te_e)

    X_train, y_train, d_train = extract_clean_window(m_train, f"{fold_id}_TRAIN")
    X_test,  y_test,  d_test  = extract_clean_window(m_test,  f"{fold_id}_TEST")

    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s  = scaler.transform(X_test)

    # Fit model
    model = LogisticRegression(**model_kwargs)
    model.fit(X_train_s, y_train)

    classes = model.classes_
    print(f"\n{fold_id} model.classes_: {classes}")

    # Predict
    pred_train = model.predict(X_train_s)
    pred_test  = model.predict(X_test_s)

    # Metrics (train + test)
    def metrics_block(y_true, y_pred, split_name):
        acc = accuracy_score(y_true, y_pred)
        f1m = f1_score(y_true, y_pred, average="macro")
        f1p = f1_score(y_true, y_pred, average=None, labels=classes)
        out = {"fold": fold_id, "split": split_name, "accuracy": acc, "macro_f1": f1m}
        for c, v in zip(classes, f1p):
            out[f"f1_class_{c}"] = v
        return out

    row_train = metrics_block(y_train, pred_train, "V2_train")
    row_test  = metrics_block(y_test,  pred_test,  "V2_test")
    metrics_rows.extend([row_train, row_test])

    cm = confusion_matrix(y_test, pred_test, labels=classes)
    cm_df = pd.DataFrame(cm, index=[f"true_{c}" for c in classes], columns=[f"pred_{c}" for c in classes])

    print(f"\n=== TEXT MIRROR: Confusion matrix ({fold_id} V2_test, labels=model.classes_) ===")
    print(cm_df.to_string())

    fig_cm = px.imshow(
        cm,
        x=[str(c) for c in classes],
        y=[str(c) for c in classes],
        text_auto=True,
        title=f"E7 (LogReg, L2) — Confusion Matrix on {fold_id} V2_test (labels=model.classes_)",
        labels=dict(x="Predicted", y="True", color="Count"),
    )
    fig_cm.show()

    # Save fold predictions (date-indexed, for Week 9)
    pred_out = PRED_DIR / f"predictions_E7_V2_{fold_id}_test.csv"
    pd.DataFrame({"date": d_test.values, "y_true": y_test, "y_pred": pred_test}).to_csv(pred_out, index=False)
    print(f"[WRITE] fold predictions saved: {pred_out}")

# Build metrics table
metrics_df = pd.DataFrame(metrics_rows)

print("\n=== TEXT MIRROR: E7 metrics (per fold, train+test rows) ===")
print(metrics_df.to_string(index=False))

test_df = metrics_df[metrics_df["split"] == "V2_test"].copy()

metric_cols = [c for c in test_df.columns if c not in ["fold", "split"]]
agg = {"fold": "AGG_MEAN", "split": "V2_test_mean"}
for c in metric_cols:
    agg[c] = float(test_df[c].mean())

agg_df = pd.DataFrame([agg])

final_metrics_df = pd.concat([metrics_df, agg_df], ignore_index=True)

print(final_metrics_df.to_string(index=False))

# Save metrics CSV
final_metrics_df.to_csv(METRICS_OUT, index=False)
print(f"\n[WRITE] metrics saved: {METRICS_OUT}")



=== Week 8 — W8.5.1 E7 (LogReg, L2, V2 folds) train/eval diagnostics ===

--- Package versions (record for determinism) ---
sklearn: 1.7.2
pandas : 2.3.3
numpy  : 2.3.5

--- Joined dataset snapshot ---
df shape: (6752, 16)
df date min/max: 2000-01-03 00:00:00 → 2025-11-25 00:00:00
df unique dates: True | df monotonic: True

--- E7 model constructor (pinned) ---
{'C': 1.0, 'penalty': 'l2', 'solver': 'lbfgs', 'multi_class': 'multinomial', 'max_iter': 200, 'class_weight': None}

=== Fold F1 ===
Train window: 2000-01-03 → 2009-12-31
Test  window: 2010-01-01 → 2011-12-31

--- F1_TRAIN extraction (DROP any NaN in F_CORE ∪ {y_L2}) ---
rows before: 2604 | dropped: 54 | after: 2550

--- F1_TEST extraction (DROP any NaN in F_CORE ∪ {y_L2}) ---
rows before: 521 | dropped: 0 | after: 521

F1 model.classes_: [-1  0  1]

=== TEXT MIRROR: Confusion matrix (F1 V2_test, labels=model.classes_) ===
         pred_-1  pred_0  pred_1
true_-1       90       0      36
true_0        27       0      67
true_1  

c:\Users\simon\Desktop\my thesis\ThesisProject\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning:

'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.



[WRITE] fold predictions saved: ..\results\predictions\predictions_E7_V2_F1_test.csv

=== Fold F2 ===
Train window: 2000-01-03 → 2011-12-31
Test  window: 2012-01-01 → 2013-12-31

--- F2_TRAIN extraction (DROP any NaN in F_CORE ∪ {y_L2}) ---
rows before: 3125 | dropped: 54 | after: 3071

--- F2_TEST extraction (DROP any NaN in F_CORE ∪ {y_L2}) ---
rows before: 522 | dropped: 0 | after: 522

F2 model.classes_: [-1  0  1]

=== TEXT MIRROR: Confusion matrix (F2 V2_test, labels=model.classes_) ===
         pred_-1  pred_0  pred_1
true_-1      134       0      43
true_0        41       0      36
true_1        44       0     224


c:\Users\simon\Desktop\my thesis\ThesisProject\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning:

'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.



[WRITE] fold predictions saved: ..\results\predictions\predictions_E7_V2_F2_test.csv

=== Fold F3 ===
Train window: 2000-01-03 → 2013-12-31
Test  window: 2014-01-01 → 2015-12-31

--- F3_TRAIN extraction (DROP any NaN in F_CORE ∪ {y_L2}) ---
rows before: 3647 | dropped: 54 | after: 3593

--- F3_TEST extraction (DROP any NaN in F_CORE ∪ {y_L2}) ---
rows before: 522 | dropped: 0 | after: 522

F3 model.classes_: [-1  0  1]

=== TEXT MIRROR: Confusion matrix (F3 V2_test, labels=model.classes_) ===
         pred_-1  pred_0  pred_1
true_-1       93       0      80
true_0        58       0      63
true_1        49       5     174


c:\Users\simon\Desktop\my thesis\ThesisProject\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning:

'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.



[WRITE] fold predictions saved: ..\results\predictions\predictions_E7_V2_F3_test.csv

=== Fold F4 ===
Train window: 2000-01-03 → 2015-12-31
Test  window: 2016-01-01 → 2019-12-31

--- F4_TRAIN extraction (DROP any NaN in F_CORE ∪ {y_L2}) ---
rows before: 4169 | dropped: 54 | after: 4115

--- F4_TEST extraction (DROP any NaN in F_CORE ∪ {y_L2}) ---
rows before: 1043 | dropped: 0 | after: 1043

F4 model.classes_: [-1  0  1]

=== TEXT MIRROR: Confusion matrix (F4 V2_test, labels=model.classes_) ===
         pred_-1  pred_0  pred_1
true_-1       50       0      97
true_0        39       0     149
true_1        35       0     673


c:\Users\simon\Desktop\my thesis\ThesisProject\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning:

'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.



[WRITE] fold predictions saved: ..\results\predictions\predictions_E7_V2_F4_test.csv

=== TEXT MIRROR: E7 metrics (per fold, train+test rows) ===
fold    split  accuracy  macro_f1  f1_class_-1  f1_class_0  f1_class_1
  F1 V2_train  0.623137  0.466846     0.645270    0.044266    0.711001
  F1  V2_test  0.687140  0.483264     0.652174    0.000000    0.797619
  F2 V2_train  0.633670  0.470635     0.642538    0.040134    0.729232
  F2  V2_test  0.685824  0.487119     0.676768    0.000000    0.784588
  F3 V2_train  0.642638  0.472587     0.645957    0.032787    0.739016
  F3  V2_test  0.511494  0.379064     0.498660    0.000000    0.638532
  F4 V2_train  0.625273  0.458824     0.622106    0.027743    0.726623
  F4  V2_test  0.693193  0.398764     0.369004    0.000000    0.827289

=== TEXT MIRROR: E7 metrics with aggregate row appended ===
    fold        split  accuracy  macro_f1  f1_class_-1  f1_class_0  f1_class_1
      F1     V2_train  0.623137  0.466846     0.645270    0.044266    0.711

In [ ]:
# W8.6.1 – E8 (LinearSVC, L2, V2): fold-by-fold scaling/train/eval + save metrics + save predictions per fold

from pathlib import Path
import pandas as pd
import numpy as np

import sklearn
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

import plotly.express as px

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 100)


# Fixed notebook-relative paths
FEATURES_PATH = Path("../data/derived/mru_usd_features_all_week5.csv")
L2_PATH       = Path("../data/derived/mru_usd_labels_L2_ma10_50_h5.csv")

# Output paths (deterministic)
METRICS_OUT = Path("../results/tables/metrics_E8.csv")
PRED_DIR = Path("../results/predictions")

if not FEATURES_PATH.exists() or not L2_PATH.exists():
    print("Not provable from provided evidence.")
    print(f"Exists features: {FEATURES_PATH.exists()} | Exists L2: {L2_PATH.exists()}")
    raise RuntimeError("STOP: missing input files")

if not METRICS_OUT.parent.exists():
    print("Not provable from provided evidence.")
    print(f"Missing metrics output directory: {METRICS_OUT.parent}")
    raise RuntimeError("STOP: missing metrics directory")

PRED_DIR.mkdir(parents=True, exist_ok=True)

# Reload (self-contained)
features_df = pd.read_csv(FEATURES_PATH)
l2_df = pd.read_csv(L2_PATH)

JOIN_KEY = "date"
LABEL_COL = "y_L2"
F_CORE = ["f_r_lag1", "f_past_ret_5", "f_past_ret_20", "f_vol_10", "f_vol_30"]

# Schema checks
missing = []
if JOIN_KEY not in features_df.columns: missing.append("features_df missing 'date'")
if JOIN_KEY not in l2_df.columns:       missing.append("l2_df missing 'date'")
if LABEL_COL not in l2_df.columns:      missing.append("l2_df missing 'y_L2'")
for c in F_CORE:
    if c not in features_df.columns:
        missing.append(f"features_df missing F_CORE '{c}'")

if missing:
    print("Not provable from provided evidence.")
    print("Schema mismatch(s):")
    for m in missing:
        print(" -", m)
    raise RuntimeError("STOP: schema mismatch")

# Parse date
features_df["date"] = pd.to_datetime(features_df["date"], errors="coerce")
l2_df["date"] = pd.to_datetime(l2_df["date"], errors="coerce")

if features_df["date"].isna().any() or l2_df["date"].isna().any():
    print("Not provable from provided evidence.")
    raise RuntimeError("STOP: date parse failure")

# Join (one-to-one)
df = pd.merge(
    features_df.sort_values("date"),
    l2_df[["date", "y_L2"]].sort_values("date"),
    on="date",
    how="inner",
    validate="one_to_one"
).sort_values("date").reset_index(drop=True)

print("\n--- Joined dataset snapshot ---")
print(f"df shape: {df.shape}")
print(f"df date min/max: {df['date'].min()} → {df['date'].max()}")
print(f"df unique dates: {df['date'].is_unique} | df monotonic: {df['date'].is_monotonic_increasing}")

# V2 folds (locked)
folds = [
    ("F1", pd.Timestamp("2000-01-03"), pd.Timestamp("2009-12-31"), pd.Timestamp("2010-01-01"), pd.Timestamp("2011-12-31")),
    ("F2", pd.Timestamp("2000-01-03"), pd.Timestamp("2011-12-31"), pd.Timestamp("2012-01-01"), pd.Timestamp("2013-12-31")),
    ("F3", pd.Timestamp("2000-01-03"), pd.Timestamp("2013-12-31"), pd.Timestamp("2014-01-01"), pd.Timestamp("2015-12-31")),
    ("F4", pd.Timestamp("2000-01-03"), pd.Timestamp("2015-12-31"), pd.Timestamp("2016-01-01"), pd.Timestamp("2019-12-31")),
]

cols_to_check = F_CORE + [LABEL_COL]

def cast_labels(y, name):
    y_ser = pd.Series(y)
    if y_ser.isna().any():
        print("Not provable from provided evidence.")
        print(f"{name} contains NaN labels after dropna; this should not happen.")
        raise RuntimeError("STOP: NaN labels remain")

    if np.issubdtype(y_ser.dtype, np.floating):
        if np.all(np.isclose(y_ser.values, np.round(y_ser.values))):
            return y_ser.round(0).astype(int).values
        print("Not provable from provided evidence.")
        print(f"{name} labels are non-integer floats; cannot safely cast.")
        raise RuntimeError("STOP: invalid label dtype")
    return y_ser.astype(int).values if np.issubdtype(y_ser.dtype, np.integer) else y_ser.values

def extract_clean_window(mask, name):
    subset = df.loc[mask, ["date"] + cols_to_check].copy()
    before = subset.shape[0]
    any_nan = subset[cols_to_check].isna().any(axis=1)
    dropped = int(any_nan.sum())
    subset = subset.loc[~any_nan].copy()
    after = subset.shape[0]

    print(f"\n--- {name} extraction (DROP any NaN in F_CORE ∪ {{y_L2}}) ---")
    print(f"rows before: {before} | dropped: {dropped} | after: {after}")
    if after == 0:
        print("Not provable from provided evidence.")
        print(f"{name} became empty after NaN filtering.")
        raise RuntimeError("STOP: empty window after dropna")

    X = subset[F_CORE].to_numpy()
    y = cast_labels(subset["y_L2"].to_numpy(), name)
    dates = subset["date"].copy()
    return X, y, dates

# Model constructor (pinned from Week 6 V2: LinearSVC max_iter=2000)
model_kwargs = dict(
    C=1.0,
    class_weight=None,
    max_iter=2000,
)

print(model_kwargs)

metrics_rows = []

for fold_id, tr_s, tr_e, te_s, te_e in folds:
    print("\n" + "="*80)
    print(f"=== Fold {fold_id} ===")
    print(f"Train window: {tr_s.date()} → {tr_e.date()}")
    print(f"Test  window: {te_s.date()} → {te_e.date()}")

    m_train = (df["date"] >= tr_s) & (df["date"] <= tr_e)
    m_test  = (df["date"] >= te_s) & (df["date"] <= te_e)

    X_train, y_train, d_train = extract_clean_window(m_train, f"{fold_id}_TRAIN")
    X_test,  y_test,  d_test  = extract_clean_window(m_test,  f"{fold_id}_TEST")

    # Scaling: train-only (per fold)
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s  = scaler.transform(X_test)

    # Fit model
    model = LinearSVC(**model_kwargs)
    model.fit(X_train_s, y_train)

    classes = model.classes_
    print(f"\n{fold_id} model.classes_: {classes}")

    # Predict
    pred_train = model.predict(X_train_s)
    pred_test  = model.predict(X_test_s)

    # Metrics (train + test)
    def metrics_block(y_true, y_pred, split_name):
        acc = accuracy_score(y_true, y_pred)
        f1m = f1_score(y_true, y_pred, average="macro")
        f1p = f1_score(y_true, y_pred, average=None, labels=classes)
        out = {"fold": fold_id, "split": split_name, "accuracy": acc, "macro_f1": f1m}
        for c, v in zip(classes, f1p):
            out[f"f1_class_{c}"] = v
        return out

    metrics_rows.append(metrics_block(y_train, pred_train, "V2_train"))
    metrics_rows.append(metrics_block(y_test,  pred_test,  "V2_test"))

    cm = confusion_matrix(y_test, pred_test, labels=classes)
    cm_df = pd.DataFrame(cm, index=[f"true_{c}" for c in classes], columns=[f"pred_{c}" for c in classes])

    print(f"\n=== TEXT MIRROR: Confusion matrix ({fold_id} V2_test, labels=model.classes_) ===")
    print(cm_df.to_string())

    fig_cm = px.imshow(
        cm,
        x=[str(c) for c in classes],
        y=[str(c) for c in classes],
        text_auto=True,
        title=f"E8 (LinearSVC, L2) — Confusion Matrix on {fold_id} V2_test (labels=model.classes_)",
        labels=dict(x="Predicted", y="True", color="Count"),
    )
    fig_cm.show()

    pred_out = PRED_DIR / f"predictions_E8_V2_{fold_id}_test.csv"
    pd.DataFrame({"date": d_test.values, "y_true": y_test, "y_pred": pred_test}).to_csv(pred_out, index=False)
    print(f"[WRITE] fold predictions saved: {pred_out}")

# Build metrics table
metrics_df = pd.DataFrame(metrics_rows)

print(metrics_df.to_string(index=False))

# Aggregate across folds (TEST only) – simple mean (unweighted)
test_df = metrics_df[metrics_df["split"] == "V2_test"].copy()
metric_cols = [c for c in test_df.columns if c not in ["fold", "split"]]
agg = {"fold": "AGG_MEAN", "split": "V2_test_mean"}
for c in metric_cols:
    agg[c] = float(test_df[c].mean())

final_metrics_df = pd.concat([metrics_df, pd.DataFrame([agg])], ignore_index=True)

print(final_metrics_df.to_string(index=False))

# Save metrics CSV
final_metrics_df.to_csv(METRICS_OUT, index=False)
print(f"\n[WRITE] metrics saved: {METRICS_OUT}")



=== Week 8 — W8.6.1 E8 (LinearSVC, L2, V2 folds) train/eval diagnostics ===

--- Package versions (record for determinism) ---
sklearn: 1.7.2
pandas : 2.3.3
numpy  : 2.3.5

--- Joined dataset snapshot ---
df shape: (6752, 16)
df date min/max: 2000-01-03 00:00:00 → 2025-11-25 00:00:00
df unique dates: True | df monotonic: True

--- E8 model constructor (pinned) ---
{'C': 1.0, 'class_weight': None, 'max_iter': 2000}

=== Fold F1 ===
Train window: 2000-01-03 → 2009-12-31
Test  window: 2010-01-01 → 2011-12-31

--- F1_TRAIN extraction (DROP any NaN in F_CORE ∪ {y_L2}) ---
rows before: 2604 | dropped: 54 | after: 2550

--- F1_TEST extraction (DROP any NaN in F_CORE ∪ {y_L2}) ---
rows before: 521 | dropped: 0 | after: 521

F1 model.classes_: [-1  0  1]

=== TEXT MIRROR: Confusion matrix (F1 V2_test, labels=model.classes_) ===
         pred_-1  pred_0  pred_1
true_-1       90       0      36
true_0        25       0      69
true_1        26       0     275


[WRITE] fold predictions saved: ..\results\predictions\predictions_E8_V2_F1_test.csv

=== Fold F2 ===
Train window: 2000-01-03 → 2011-12-31
Test  window: 2012-01-01 → 2013-12-31

--- F2_TRAIN extraction (DROP any NaN in F_CORE ∪ {y_L2}) ---
rows before: 3125 | dropped: 54 | after: 3071

--- F2_TEST extraction (DROP any NaN in F_CORE ∪ {y_L2}) ---
rows before: 522 | dropped: 0 | after: 522

F2 model.classes_: [-1  0  1]

=== TEXT MIRROR: Confusion matrix (F2 V2_test, labels=model.classes_) ===
         pred_-1  pred_0  pred_1
true_-1      119       0      58
true_0        32       0      45
true_1        37       0     231


[WRITE] fold predictions saved: ..\results\predictions\predictions_E8_V2_F2_test.csv

=== Fold F3 ===
Train window: 2000-01-03 → 2013-12-31
Test  window: 2014-01-01 → 2015-12-31

--- F3_TRAIN extraction (DROP any NaN in F_CORE ∪ {y_L2}) ---
rows before: 3647 | dropped: 54 | after: 3593

--- F3_TEST extraction (DROP any NaN in F_CORE ∪ {y_L2}) ---
rows before: 522 | dropped: 0 | after: 522

F3 model.classes_: [-1  0  1]

=== TEXT MIRROR: Confusion matrix (F3 V2_test, labels=model.classes_) ===
         pred_-1  pred_0  pred_1
true_-1       77       0      96
true_0        55       0      66
true_1        46       0     182


[WRITE] fold predictions saved: ..\results\predictions\predictions_E8_V2_F3_test.csv

=== Fold F4 ===
Train window: 2000-01-03 → 2015-12-31
Test  window: 2016-01-01 → 2019-12-31

--- F4_TRAIN extraction (DROP any NaN in F_CORE ∪ {y_L2}) ---
rows before: 4169 | dropped: 54 | after: 4115

--- F4_TEST extraction (DROP any NaN in F_CORE ∪ {y_L2}) ---
rows before: 1043 | dropped: 0 | after: 1043

F4 model.classes_: [-1  0  1]

=== TEXT MIRROR: Confusion matrix (F4 V2_test, labels=model.classes_) ===
         pred_-1  pred_0  pred_1
true_-1       45       0     102
true_0        32       0     156
true_1        19       0     689


[WRITE] fold predictions saved: ..\results\predictions\predictions_E8_V2_F4_test.csv

=== TEXT MIRROR: E8 metrics (per fold, train+test rows) ===
fold    split  accuracy  macro_f1  f1_class_-1  f1_class_0  f1_class_1
  F1 V2_train  0.618039  0.448474     0.635227    0.000000    0.710196
  F1  V2_test  0.700576  0.493931     0.674157    0.000000    0.807636
  F2 V2_train  0.636600  0.463069     0.651163    0.010471    0.727573
  F2  V2_test  0.670498  0.473166     0.652055    0.000000    0.767442
  F3 V2_train  0.642638  0.461833     0.650246    0.000000    0.735253
  F3  V2_test  0.496169  0.358370     0.438746    0.000000    0.636364
  F4 V2_train  0.625273  0.455614     0.621762    0.023107    0.721975
  F4  V2_test  0.703739  0.401000     0.370370    0.000000    0.832628

=== TEXT MIRROR: E8 metrics with aggregate row appended ===
    fold        split  accuracy  macro_f1  f1_class_-1  f1_class_0  f1_class_1
      F1     V2_train  0.618039  0.448474     0.635227    0.000000    0.710

In [ ]:
# W8.7.1 – Artifact audit: verify metrics_E1–E8 + predictions files exist (fail-fast inventory)

from pathlib import Path


ROOT_RESULTS = Path("../results")
TABLES = ROOT_RESULTS / "tables"
PREDS  = ROOT_RESULTS / "predictions"

expected_metrics = [
    TABLES / "metrics_E1.csv",
    TABLES / "metrics_E2.csv",
    TABLES / "metrics_E3.csv",
    TABLES / "metrics_E4.csv",
    TABLES / "metrics_E5.csv",
    TABLES / "metrics_E6.csv",
    TABLES / "metrics_E7.csv",
    TABLES / "metrics_E8.csv",
]

expected_preds = [
    # V1 test predictions (already written for E5/E6; E1–E4 )
    PREDS / "predictions_E1_V1_test.csv",
    PREDS / "predictions_E2_V1_test.csv",
    PREDS / "predictions_E3_V1_test.csv",
    PREDS / "predictions_E4_V1_test.csv",
    PREDS / "predictions_E5_V1_test.csv",
    PREDS / "predictions_E6_V1_test.csv",
    # V2 fold test predictions (already written for E7/E8; E2/E4 )
    PREDS / "predictions_E2_V2_F1_test.csv",
    PREDS / "predictions_E2_V2_F2_test.csv",
    PREDS / "predictions_E2_V2_F3_test.csv",
    PREDS / "predictions_E2_V2_F4_test.csv",
    PREDS / "predictions_E4_V2_F1_test.csv",
    PREDS / "predictions_E4_V2_F2_test.csv",
    PREDS / "predictions_E4_V2_F3_test.csv",
    PREDS / "predictions_E4_V2_F4_test.csv",
    PREDS / "predictions_E7_V2_F1_test.csv",
    PREDS / "predictions_E7_V2_F2_test.csv",
    PREDS / "predictions_E7_V2_F3_test.csv",
    PREDS / "predictions_E7_V2_F4_test.csv",
    PREDS / "predictions_E8_V2_F1_test.csv",
    PREDS / "predictions_E8_V2_F2_test.csv",
    PREDS / "predictions_E8_V2_F3_test.csv",
    PREDS / "predictions_E8_V2_F4_test.csv",
]

print(f"tables dir: {TABLES} | exists: {TABLES.exists()}")
print(f"preds  dir: {PREDS}  | exists: {PREDS.exists()}")

print("\n--- Metrics files check ---")
missing_metrics = []
for p in expected_metrics:
    ok = p.exists()
    print(f"{'OK ' if ok else 'MISSING'}  {p}")
    if not ok:
        missing_metrics.append(str(p))

print("\n--- Predictions files check ---")
missing_preds = []
for p in expected_preds:
    ok = p.exists()
    print(f"{'OK ' if ok else 'MISSING'}  {p}")
    if not ok:
        missing_preds.append(str(p))

print("\n=== SUMMARY ===")
print(f"Missing metrics count: {len(missing_metrics)}")
print(f"Missing preds count:   {len(missing_preds)}")

if missing_metrics or missing_preds:
    print("\nSTOP: Artifact set incomplete.")
    print("\nMissing metrics:")
    for x in missing_metrics:
        print(" -", x)
    print("\nMissing preds:")
    for x in missing_preds:
        print(" -", x)



=== Week 8 — W8.7.1 Artifact audit (metrics + predictions) ===

--- Directory existence ---
tables dir: ..\results\tables | exists: True
preds  dir: ..\results\predictions  | exists: True

--- Metrics files check ---
OK   ..\results\tables\metrics_E1.csv
OK   ..\results\tables\metrics_E2.csv
OK   ..\results\tables\metrics_E3.csv
OK   ..\results\tables\metrics_E4.csv
OK   ..\results\tables\metrics_E5.csv
OK   ..\results\tables\metrics_E6.csv
OK   ..\results\tables\metrics_E7.csv
OK   ..\results\tables\metrics_E8.csv

--- Predictions files check ---
MISSING  ..\results\predictions\predictions_E1_V1_test.csv
MISSING  ..\results\predictions\predictions_E2_V1_test.csv
MISSING  ..\results\predictions\predictions_E3_V1_test.csv
MISSING  ..\results\predictions\predictions_E4_V1_test.csv
OK   ..\results\predictions\predictions_E5_V1_test.csv
OK   ..\results\predictions\predictions_E6_V1_test.csv
MISSING  ..\results\predictions\predictions_E2_V2_F1_test.csv
MISSING  ..\results\predictions\predic

In [ ]:
# W8.7.2 – Generate missing prediction files for E1–E4 (L1_Q) deterministically (V1 + V2 folds)

from pathlib import Path
import pandas as pd
import numpy as np

import sklearn
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 120)


FEATURES_PATH = Path("../data/derived/mru_usd_features_all_week5.csv")
L1Q_PATH      = Path("../data/derived/mru_usd_labels_L1_Q_k5_q20_60_20.csv")

# Outputs
PRED_DIR = Path("../results/predictions")
PRED_DIR.mkdir(parents=True, exist_ok=True)

targets_to_write = [
    PRED_DIR / "predictions_E1_V1_test.csv",
    PRED_DIR / "predictions_E2_V1_test.csv",
    PRED_DIR / "predictions_E3_V1_test.csv",
    PRED_DIR / "predictions_E4_V1_test.csv",
    PRED_DIR / "predictions_E2_V2_F1_test.csv",
    PRED_DIR / "predictions_E2_V2_F2_test.csv",
    PRED_DIR / "predictions_E2_V2_F3_test.csv",
    PRED_DIR / "predictions_E2_V2_F4_test.csv",
    PRED_DIR / "predictions_E4_V2_F1_test.csv",
    PRED_DIR / "predictions_E4_V2_F2_test.csv",
    PRED_DIR / "predictions_E4_V2_F3_test.csv",
    PRED_DIR / "predictions_E4_V2_F4_test.csv",
]

# Quick existence check
print("\n--- Target files (will write only if missing) ---")
for p in targets_to_write:
    print(f"{'MISSING' if not p.exists() else 'EXISTS '}  {p}")

if not FEATURES_PATH.exists() or not L1Q_PATH.exists():
    print("Not provable from provided evidence.")
    print(f"Exists features: {FEATURES_PATH.exists()} | Exists L1_Q labels: {L1Q_PATH.exists()}")
    raise RuntimeError("STOP: missing input files")

# Load CSVs
features_df = pd.read_csv(FEATURES_PATH)
l1q_df = pd.read_csv(L1Q_PATH)

# Required schema
JOIN_KEY = "date"
F_CORE = ["f_r_lag1", "f_past_ret_5", "f_past_ret_20", "f_vol_10", "f_vol_30"]

# Schema checks
missing = []
if JOIN_KEY not in features_df.columns: missing.append("features_df missing 'date'")
if JOIN_KEY not in l1q_df.columns:      missing.append("l1q_df missing 'date'")
for c in F_CORE:
    if c not in features_df.columns:
        missing.append(f"features_df missing F_CORE '{c}'")

if missing:
    print("Not provable from provided evidence.")
    print("Schema mismatch(s):")
    for m in missing:
        print(" -", m)
    raise RuntimeError("STOP: schema mismatch")

# Parse date
features_df["date"] = pd.to_datetime(features_df["date"], errors="coerce")
l1q_df["date"] = pd.to_datetime(l1q_df["date"], errors="coerce")

if features_df["date"].isna().any() or l1q_df["date"].isna().any():
    print("Not provable from provided evidence.")
    raise RuntimeError("STOP: date parse failure")


label_priority = [
    "y_L1_Q", "y_l1_q",
    "y_L1Q", "y_l1q",
    "y_L1", "y_l1",
    "label", "y"
]
found = [c for c in label_priority if c in l1q_df.columns]

if len(found) == 0:
    print("Not provable from provided evidence.")
    print("Could not identify the L1_Q label column name from the label CSV.")
    print("Available columns in L1_Q labels file:")
    print(list(l1q_df.columns))
    raise RuntimeError("STOP: unknown label column")

LABEL_COL = found[0]
print(f"\n--- L1_Q label column selected: {LABEL_COL} ---")

df = pd.merge(
    features_df.sort_values("date"),
    l1q_df.sort_values("date"),
    on="date",
    how="inner",
    validate="one_to_one"
).sort_values("date").reset_index(drop=True)

print(f"df shape: {df.shape}")
print(f"df date min/max: {df['date'].min()} → {df['date'].max()}")
print(f"df unique dates: {df['date'].is_unique} | df monotonic: {df['date'].is_monotonic_increasing}")

# Split boundaries (V1 locked)
V1_TRAIN_START = pd.Timestamp("2000-01-03")
V1_TRAIN_END   = pd.Timestamp("2015-12-31")
V1_VAL_START   = pd.Timestamp("2016-01-01")
V1_VAL_END     = pd.Timestamp("2019-12-31")
V1_TEST_START  = pd.Timestamp("2020-01-01")
V1_TEST_END    = pd.Timestamp("2025-11-25")

# V2 folds (locked)
folds = [
    ("F1", pd.Timestamp("2000-01-03"), pd.Timestamp("2009-12-31"), pd.Timestamp("2010-01-01"), pd.Timestamp("2011-12-31")),
    ("F2", pd.Timestamp("2000-01-03"), pd.Timestamp("2011-12-31"), pd.Timestamp("2012-01-01"), pd.Timestamp("2013-12-31")),
    ("F3", pd.Timestamp("2000-01-03"), pd.Timestamp("2013-12-31"), pd.Timestamp("2014-01-01"), pd.Timestamp("2015-12-31")),
    ("F4", pd.Timestamp("2000-01-03"), pd.Timestamp("2015-12-31"), pd.Timestamp("2016-01-01"), pd.Timestamp("2019-12-31")),
]

cols_to_check = F_CORE + [LABEL_COL]

def cast_labels(y, name):
    y_ser = pd.Series(y)
    if y_ser.isna().any():
        print("Not provable from provided evidence.")
        print(f"{name} contains NaN labels after dropna; this should not happen.")
        raise RuntimeError("STOP: NaN labels remain")

    if np.issubdtype(y_ser.dtype, np.floating):
        if np.all(np.isclose(y_ser.values, np.round(y_ser.values))):
            return y_ser.round(0).astype(int).values
        print("Not provable from provided evidence.")
        print(f"{name} labels are non-integer floats; cannot safely cast.")
        raise RuntimeError("STOP: invalid label dtype")
    if np.issubdtype(y_ser.dtype, np.integer):
        return y_ser.astype(int).values
    try:
        return y_ser.astype(int).values
    except Exception:
        print("Not provable from provided evidence.")
        print(f"{name} label dtype cannot be converted to int safely: {y_ser.dtype}")
        raise RuntimeError("STOP: label cast failure")

def extract_clean(mask, window_name):
    subset = df.loc[mask, ["date"] + cols_to_check].copy()
    before = subset.shape[0]
    any_nan = subset[cols_to_check].isna().any(axis=1)
    dropped = int(any_nan.sum())
    subset = subset.loc[~any_nan].copy()
    after = subset.shape[0]
    if after == 0:
        print("Not provable from provided evidence.")
        print(f"{window_name} became empty after DROP any NaN in (F_CORE ∪ {{label}}).")
        raise RuntimeError("STOP: empty window")
    X = subset[F_CORE].to_numpy()
    y = cast_labels(subset[LABEL_COL].to_numpy(), window_name)
    d = subset["date"].copy()
    return X, y, d, before, dropped, after

def fit_predict_and_write(model, X_tr, y_tr, X_te, y_te, d_te, out_path, tag):
    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_tr)
    X_te_s = scaler.transform(X_te)
    model.fit(X_tr_s, y_tr)
    y_pred = model.predict(X_te_s)
    pd.DataFrame({"date": d_te.values, "y_true": y_te, "y_pred": y_pred}).to_csv(out_path, index=False)
    print(f"[WRITE] {tag}: {out_path}")


m_v1_train = (df["date"] >= V1_TRAIN_START) & (df["date"] <= V1_TRAIN_END)
m_v1_test  = (df["date"] >= V1_TEST_START)  & (df["date"] <= V1_TEST_END)

X_tr, y_tr, d_tr, b1, dr1, a1 = extract_clean(m_v1_train, "V1_TRAIN")
X_te, y_te, d_te, b2, dr2, a2 = extract_clean(m_v1_test,  "V1_TEST")

print("\n--- V1 extraction summary (L1_Q) ---")
print(f"V1_TRAIN: before={b1} dropped={dr1} after={a1} | date min/max={d_tr.min()} → {d_tr.max()}")
print(f"V1_TEST : before={b2} dropped={dr2} after={a2} | date min/max={d_te.min()} → {d_te.max()}")

v1_targets = {
    "E1": PRED_DIR / "predictions_E1_V1_test.csv",
    "E2": PRED_DIR / "predictions_E2_V1_test.csv",
    "E3": PRED_DIR / "predictions_E3_V1_test.csv",
    "E4": PRED_DIR / "predictions_E4_V1_test.csv",
}

if not v1_targets["E1"].exists():
    model_E1 = LogisticRegression(C=1.0, penalty="l2", solver="lbfgs", multi_class="auto", max_iter=1000, class_weight=None)
    fit_predict_and_write(model_E1, X_tr, y_tr, X_te, y_te, d_te, v1_targets["E1"], "E1 V1_test")

if not v1_targets["E2"].exists():
    model_E2_v1 = LogisticRegression(C=1.0, penalty="l2", solver="lbfgs", multi_class="multinomial", max_iter=200, class_weight=None)
    fit_predict_and_write(model_E2_v1, X_tr, y_tr, X_te, y_te, d_te, v1_targets["E2"], "E2 V1_test")

if not v1_targets["E3"].exists():
    model_E3 = LinearSVC(C=1.0, class_weight=None, max_iter=5000)
    fit_predict_and_write(model_E3, X_tr, y_tr, X_te, y_te, d_te, v1_targets["E3"], "E3 V1_test")

if not v1_targets["E4"].exists():
    model_E4_v1 = LinearSVC(C=1.0, class_weight=None, max_iter=2000)
    fit_predict_and_write(model_E4_v1, X_tr, y_tr, X_te, y_te, d_te, v1_targets["E4"], "E4 V1_test")



for fold_id, tr_s, tr_e, te_s, te_e in folds:
    m_tr = (df["date"] >= tr_s) & (df["date"] <= tr_e)
    m_te = (df["date"] >= te_s) & (df["date"] <= te_e)

    X_tr, y_tr, d_tr, b1, dr1, a1 = extract_clean(m_tr, f"{fold_id}_TRAIN")
    X_te, y_te, d_te, b2, dr2, a2 = extract_clean(m_te, f"{fold_id}_TEST")

    print(f"\nFold {fold_id}: TRAIN before={b1} dropped={dr1} after={a1} | TEST before={b2} dropped={dr2} after={a2}")

    out_E2 = PRED_DIR / f"predictions_E2_V2_{fold_id}_test.csv"
    out_E4 = PRED_DIR / f"predictions_E4_V2_{fold_id}_test.csv"

    if not out_E2.exists():
        model_E2 = LogisticRegression(C=1.0, penalty="l2", solver="lbfgs", multi_class="multinomial", max_iter=200, class_weight=None)
        fit_predict_and_write(model_E2, X_tr, y_tr, X_te, y_te, d_te, out_E2, f"E2 V2_{fold_id}_test")

    if not out_E4.exists():
        model_E4 = LinearSVC(C=1.0, class_weight=None, max_iter=2000)
        fit_predict_and_write(model_E4, X_tr, y_tr, X_te, y_te, d_te, out_E4, f"E4 V2_{fold_id}_test")



=== Week 8 — W8.7.2 Generate missing predictions for E1–E4 (L1_Q) ===

--- Package versions (record for determinism) ---
sklearn: 1.7.2
pandas : 2.3.3
numpy  : 2.3.5

--- Target files (will write only if missing) ---
MISSING  ..\results\predictions\predictions_E1_V1_test.csv
MISSING  ..\results\predictions\predictions_E2_V1_test.csv
MISSING  ..\results\predictions\predictions_E3_V1_test.csv
MISSING  ..\results\predictions\predictions_E4_V1_test.csv
MISSING  ..\results\predictions\predictions_E2_V2_F1_test.csv
MISSING  ..\results\predictions\predictions_E2_V2_F2_test.csv
MISSING  ..\results\predictions\predictions_E2_V2_F3_test.csv
MISSING  ..\results\predictions\predictions_E2_V2_F4_test.csv
MISSING  ..\results\predictions\predictions_E4_V2_F1_test.csv
MISSING  ..\results\predictions\predictions_E4_V2_F2_test.csv
MISSING  ..\results\predictions\predictions_E4_V2_F3_test.csv
MISSING  ..\results\predictions\predictions_E4_V2_F4_test.csv

--- L1_Q label column selected: y_L1_Q ---

--- Jo

c:\Users\simon\Desktop\my thesis\ThesisProject\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning:

'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.

c:\Users\simon\Desktop\my thesis\ThesisProject\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning:

'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.

c:\Users\simon\Desktop\my thesis\ThesisProject\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning:

'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.

c:\Users\simon\Desktop\my thesis\ThesisProject\.venv\Lib\site-packages\sklearn\linear_model\_logi

In [ ]:

from pathlib import Path


ROOT_RESULTS = Path("../results")
TABLES = ROOT_RESULTS / "tables"
PREDS  = ROOT_RESULTS / "predictions"

expected_metrics = [
    TABLES / "metrics_E1.csv",
    TABLES / "metrics_E2.csv",
    TABLES / "metrics_E3.csv",
    TABLES / "metrics_E4.csv",
    TABLES / "metrics_E5.csv",
    TABLES / "metrics_E6.csv",
    TABLES / "metrics_E7.csv",
    TABLES / "metrics_E8.csv",
]

expected_preds = [
    # V1 test predictions
    PREDS / "predictions_E1_V1_test.csv",
    PREDS / "predictions_E2_V1_test.csv",
    PREDS / "predictions_E3_V1_test.csv",
    PREDS / "predictions_E4_V1_test.csv",
    PREDS / "predictions_E5_V1_test.csv",
    PREDS / "predictions_E6_V1_test.csv",

    # V2 fold test predictions
    PREDS / "predictions_E2_V2_F1_test.csv",
    PREDS / "predictions_E2_V2_F2_test.csv",
    PREDS / "predictions_E2_V2_F3_test.csv",
    PREDS / "predictions_E2_V2_F4_test.csv",

    PREDS / "predictions_E4_V2_F1_test.csv",
    PREDS / "predictions_E4_V2_F2_test.csv",
    PREDS / "predictions_E4_V2_F3_test.csv",
    PREDS / "predictions_E4_V2_F4_test.csv",

    PREDS / "predictions_E7_V2_F1_test.csv",
    PREDS / "predictions_E7_V2_F2_test.csv",
    PREDS / "predictions_E7_V2_F3_test.csv",
    PREDS / "predictions_E7_V2_F4_test.csv",

    PREDS / "predictions_E8_V2_F1_test.csv",
    PREDS / "predictions_E8_V2_F2_test.csv",
    PREDS / "predictions_E8_V2_F3_test.csv",
    PREDS / "predictions_E8_V2_F4_test.csv",
]

print("\n--- Directory existence ---")
print(f"tables dir: {TABLES} | exists: {TABLES.exists()}")
print(f"preds  dir: {PREDS}  | exists: {PREDS.exists()}")

print("\n--- Metrics files check ---")
missing_metrics = []
for p in expected_metrics:
    ok = p.exists()
    print(f"{'OK ' if ok else 'MISSING'}  {p}")
    if not ok:
        missing_metrics.append(str(p))

print("\n--- Predictions files check ---")
missing_preds = []
for p in expected_preds:
    ok = p.exists()
    print(f"{'OK ' if ok else 'MISSING'}  {p}")
    if not ok:
        missing_preds.append(str(p))

print("\n=== ACCEPTANCE GATE ===")
if len(missing_metrics) == 0:
    print("[PASS] All metrics_E1–E8 present")
else:
    print("[FAIL] Missing metrics files:", len(missing_metrics))

if len(missing_preds) == 0:
    print("[PASS] All required prediction files present (V1 + V2)")
else:
    print("[FAIL] Missing prediction files:", len(missing_preds))

if missing_metrics or missing_preds:
    print("\nSTOP: Artifact set still incomplete.")
    if missing_metrics:
        print("\nMissing metrics:")
        for x in missing_metrics:
            print(" -", x)
    if missing_preds:
        print("\nMissing preds:")
        for x in missing_preds:
            print(" -", x)



=== Week 8 — W8.7.3 Artifact audit (post-generation) ===

--- Directory existence ---
tables dir: ..\results\tables | exists: True
preds  dir: ..\results\predictions  | exists: True

--- Metrics files check ---
OK   ..\results\tables\metrics_E1.csv
OK   ..\results\tables\metrics_E2.csv
OK   ..\results\tables\metrics_E3.csv
OK   ..\results\tables\metrics_E4.csv
OK   ..\results\tables\metrics_E5.csv
OK   ..\results\tables\metrics_E6.csv
OK   ..\results\tables\metrics_E7.csv
OK   ..\results\tables\metrics_E8.csv

--- Predictions files check ---
OK   ..\results\predictions\predictions_E1_V1_test.csv
OK   ..\results\predictions\predictions_E2_V1_test.csv
OK   ..\results\predictions\predictions_E3_V1_test.csv
OK   ..\results\predictions\predictions_E4_V1_test.csv
OK   ..\results\predictions\predictions_E5_V1_test.csv
OK   ..\results\predictions\predictions_E6_V1_test.csv
OK   ..\results\predictions\predictions_E2_V2_F1_test.csv
OK   ..\results\predictions\predictions_E2_V2_F2_test.csv
OK   .

In [ ]:
# W8.8.1  — Load & verify artifacts (metrics + predictions)

from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 200)


RESULTS = Path("../results")
TABLES  = RESULTS / "tables"
PREDS   = RESULTS / "predictions"
FIGS    = RESULTS / "figures"

for d in [RESULTS, TABLES, PREDS, FIGS]:
    if not d.exists():
        print("Not provable from provided evidence.")
        print(f"Missing required directory: {d}")
        raise RuntimeError("STOP: missing results directory")

metrics_files = sorted(TABLES.glob("metrics_E*.csv"))
pred_files    = sorted(PREDS.glob("predictions_*.csv"))

print(f"Found metrics files: {len(metrics_files)}")
for p in metrics_files:
    print(" -", p)

print(f"\nFound prediction files: {len(pred_files)}")
for p in pred_files:
    print(" -", p)

expected_metrics = [TABLES / f"metrics_E{i}.csv" for i in range(1, 9)]

expected_preds = [
    # V1 test
    PREDS / "predictions_E1_V1_test.csv",
    PREDS / "predictions_E2_V1_test.csv",
    PREDS / "predictions_E3_V1_test.csv",
    PREDS / "predictions_E4_V1_test.csv",
    PREDS / "predictions_E5_V1_test.csv",
    PREDS / "predictions_E6_V1_test.csv",
    # V2 folds
    PREDS / "predictions_E2_V2_F1_test.csv",
    PREDS / "predictions_E2_V2_F2_test.csv",
    PREDS / "predictions_E2_V2_F3_test.csv",
    PREDS / "predictions_E2_V2_F4_test.csv",
    PREDS / "predictions_E4_V2_F1_test.csv",
    PREDS / "predictions_E4_V2_F2_test.csv",
    PREDS / "predictions_E4_V2_F3_test.csv",
    PREDS / "predictions_E4_V2_F4_test.csv",
    PREDS / "predictions_E7_V2_F1_test.csv",
    PREDS / "predictions_E7_V2_F2_test.csv",
    PREDS / "predictions_E7_V2_F3_test.csv",
    PREDS / "predictions_E7_V2_F4_test.csv",
    PREDS / "predictions_E8_V2_F1_test.csv",
    PREDS / "predictions_E8_V2_F2_test.csv",
    PREDS / "predictions_E8_V2_F3_test.csv",
    PREDS / "predictions_E8_V2_F4_test.csv",
]

missing = [str(p) for p in expected_metrics + expected_preds if not p.exists()]
if missing:
    print("Not provable from provided evidence.")
    print("Missing required artifacts:")
    for m in missing:
        print(" -", m)
    raise RuntimeError("STOP: missing required files")

LABELS = [-1, 0, 1]

def parse_experiment_id(path: Path) -> str:
    name = path.name
    if "metrics_E" in name:
        return name.split("metrics_")[1].split(".csv")[0]   # E1..E8
    if "predictions_E" in name:
        return name.split("predictions_")[1].split("_")[0]  # E1..E8
    return "UNKNOWN"

def coerce_date(s: pd.Series, path: Path) -> pd.Series:
    dt = pd.to_datetime(s, errors="coerce")
    if dt.isna().any():
        bad = int(dt.isna().sum())
        print("Not provable from provided evidence.")
        print(f"{path} has {bad} rows where 'date' failed datetime parsing.")
        raise RuntimeError("STOP: bad date parsing")
    return dt

def check_label_space(series: pd.Series, path: Path, col: str):
    x = series.dropna()
    if x.empty:
        print("Not provable from provided evidence.")
        print(f"{path} column '{col}' is empty after dropping NaNs.")
        raise RuntimeError("STOP: empty label column")

    vals = np.array(x)
    if np.issubdtype(vals.dtype, np.floating):
        if not np.all(np.isclose(vals, np.round(vals))):
            print("Not provable from provided evidence.")
            print(f"{path} column '{col}' contains non-integer floats.")
            print("Distinct values (sample):", sorted(pd.unique(x))[:20])
            raise RuntimeError("STOP: invalid label values")
        vals = np.round(vals).astype(int)
    else:
        vals = vals.astype(int)

    uniq = sorted(set(vals.tolist()))
    if not set(uniq).issubset(set(LABELS)):
        print("Not provable from provided evidence.")
        print(f"{path} column '{col}' has labels outside {LABELS}: {uniq}")
        raise RuntimeError("STOP: label-space mismatch")
    return uniq

# Predictions verification table
pred_rows = []
for p in expected_preds:
    df = pd.read_csv(p)
    required = {"date", "y_true", "y_pred"}
    if not required.issubset(set(df.columns)):
        print("Not provable from provided evidence.")
        print(f"{p} missing required columns. Found: {list(df.columns)}")
        raise RuntimeError("STOP: malformed predictions CSV")

    df["date"] = coerce_date(df["date"], p)

    uniq_true = check_label_space(df["y_true"], p, "y_true")
    uniq_pred = check_label_space(df["y_pred"], p, "y_pred")

    pred_rows.append({
        "experiment": parse_experiment_id(p),
        "file": str(p),
        "rows": int(df.shape[0]),
        "date_min": str(df["date"].min()),
        "date_max": str(df["date"].max()),
        "uniq_y_true": str(uniq_true),
        "uniq_y_pred": str(uniq_pred),
        "columns": str(list(df.columns)),
    })

pred_verif = pd.DataFrame(pred_rows).sort_values(["experiment", "file"]).reset_index(drop=True)

metrics_rows = []
for p in expected_metrics:
    df = pd.read_csv(p)
    needed_min = {"split", "accuracy", "macro_f1"}
    if not needed_min.issubset(set(df.columns)):
        print("Not provable from provided evidence.")
        print(f"{p} missing required minimal metric columns {sorted(needed_min)}.")
        print("Found:", list(df.columns))
        raise RuntimeError("STOP: malformed metrics CSV")

    has_per_class = all(c in df.columns for c in ["f1_class_-1", "f1_class_0", "f1_class_1"])

    metrics_rows.append({
        "experiment": parse_experiment_id(p),
        "file": str(p),
        "rows": int(df.shape[0]),
        "split_values": str(sorted(pd.unique(df["split"]))),
        "has_f1_class_cols": bool(has_per_class),
        "columns": str(list(df.columns)),
    })

metrics_verif = pd.DataFrame(metrics_rows).sort_values(["experiment"]).reset_index(drop=True)

print(pred_verif[["experiment", "rows", "date_min", "date_max", "uniq_y_pred", "file"]].to_string(index=False))

print(metrics_verif[["experiment", "rows", "split_values", "has_f1_class_cols", "file"]].to_string(index=False))

missing_class_f1 = metrics_verif.loc[~metrics_verif["has_f1_class_cols"], "experiment"].tolist()


=== Week 8 — W8.8.1 (FIXED) Load & verify artifacts (metrics + predictions) ===

--- Programmatic file listing ---
Found metrics files: 8
 - ..\results\tables\metrics_E1.csv
 - ..\results\tables\metrics_E2.csv
 - ..\results\tables\metrics_E3.csv
 - ..\results\tables\metrics_E4.csv
 - ..\results\tables\metrics_E5.csv
 - ..\results\tables\metrics_E6.csv
 - ..\results\tables\metrics_E7.csv
 - ..\results\tables\metrics_E8.csv

Found prediction files: 22
 - ..\results\predictions\predictions_E1_V1_test.csv
 - ..\results\predictions\predictions_E2_V1_test.csv
 - ..\results\predictions\predictions_E2_V2_F1_test.csv
 - ..\results\predictions\predictions_E2_V2_F2_test.csv
 - ..\results\predictions\predictions_E2_V2_F3_test.csv
 - ..\results\predictions\predictions_E2_V2_F4_test.csv
 - ..\results\predictions\predictions_E3_V1_test.csv
 - ..\results\predictions\predictions_E4_V1_test.csv
 - ..\results\predictions\predictions_E4_V2_F1_test.csv
 - ..\results\predictions\predictions_E4_V2_F2_test.cs

In [ ]:
# W8.8.2  Build summary_E5_E8_vs_E1_E4.csv from predictions

from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

pd.set_option("display.width", 190)
pd.set_option("display.max_columns", 220)


TABLES = Path("../results/tables")
PREDS  = Path("../results/predictions")
OUT    = TABLES / "summary_E5_E8_vs_E1_E4.csv"

for d in [TABLES, PREDS]:
    if not d.exists():
        print("Not provable from provided evidence.")
        print(f"Missing directory: {d}")
        raise RuntimeError("STOP: missing required directory")

LABELS = [-1, 0, 1]

META = {
    "E1": ("L1_Q", "Logistic",    "V1"),
    "E2": ("L1_Q", "Logistic",    "V2"),
    "E3": ("L1_Q", "Linear SVM",  "V1"),
    "E4": ("L1_Q", "Linear SVM",  "V2"),
    "E5": ("L2",   "Logistic",    "V1"),
    "E6": ("L2",   "Linear SVM",  "V1"),
    "E7": ("L2",   "Logistic",    "V2"),
    "E8": ("L2",   "Linear SVM",  "V2"),
}

# Test-only prediction sources (frozen)
PRED_SOURCES = {
    "E1": [PREDS / "predictions_E1_V1_test.csv"],
    "E3": [PREDS / "predictions_E3_V1_test.csv"],
    "E5": [PREDS / "predictions_E5_V1_test.csv"],
    "E6": [PREDS / "predictions_E6_V1_test.csv"],

    "E2": [
        PREDS / "predictions_E2_V2_F1_test.csv",
        PREDS / "predictions_E2_V2_F2_test.csv",
        PREDS / "predictions_E2_V2_F3_test.csv",
        PREDS / "predictions_E2_V2_F4_test.csv",
    ],
    "E4": [
        PREDS / "predictions_E4_V2_F1_test.csv",
        PREDS / "predictions_E4_V2_F2_test.csv",
        PREDS / "predictions_E4_V2_F3_test.csv",
        PREDS / "predictions_E4_V2_F4_test.csv",
    ],
    "E7": [
        PREDS / "predictions_E7_V2_F1_test.csv",
        PREDS / "predictions_E7_V2_F2_test.csv",
        PREDS / "predictions_E7_V2_F3_test.csv",
        PREDS / "predictions_E7_V2_F4_test.csv",
    ],
    "E8": [
        PREDS / "predictions_E8_V2_F1_test.csv",
        PREDS / "predictions_E8_V2_F2_test.csv",
        PREDS / "predictions_E8_V2_F3_test.csv",
        PREDS / "predictions_E8_V2_F4_test.csv",
    ],
}

METRICS_PATHS = {f"E{i}": TABLES / f"metrics_E{i}.csv" for i in range(1, 9)}

def load_pred_file(p: Path) -> pd.DataFrame:
    if not p.exists():
        print("Not provable from provided evidence.")
        print(f"Missing predictions file: {p}")
        raise RuntimeError("STOP: missing predictions file")

    df = pd.read_csv(p)
    req = {"date", "y_true", "y_pred"}
    if not req.issubset(df.columns):
        print("Not provable from provided evidence.")
        print(f"{p} missing required columns {sorted(req)}. Found: {list(df.columns)}")
        raise RuntimeError("STOP: malformed predictions CSV")

    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    if df["date"].isna().any():
        bad = int(df["date"].isna().sum())
        print("Not provable from provided evidence.")
        print(f"{p} has {bad} rows where 'date' failed datetime parsing.")
        raise RuntimeError("STOP: bad date parsing")

    for col in ["y_true", "y_pred"]:
        x = df[col].dropna().to_numpy()
        if x.size == 0:
            print("Not provable from provided evidence.")
            print(f"{p}: column '{col}' is empty after dropping NaNs.")
            raise RuntimeError("STOP: empty label column")

        if np.issubdtype(x.dtype, np.floating):
            if not np.all(np.isclose(x, np.round(x))):
                print("Not provable from provided evidence.")
                print(f"{p}: column '{col}' contains non-integer float labels.")
                print("Distinct values (sample):", sorted(pd.unique(df[col].dropna()))[:20])
                raise RuntimeError("STOP: invalid label values")
            df[col] = np.round(df[col]).astype(int)
        else:
            df[col] = df[col].astype(int)

        uniq = sorted(pd.unique(df[col]).tolist())
        if not set(uniq).issubset(set(LABELS)):
            print("Not provable from provided evidence.")
            print(f"{p}: column '{col}' has labels outside {LABELS}: {uniq}")
            raise RuntimeError("STOP: label-space mismatch")

    return df

def compute_metrics_from_df(df: pd.DataFrame) -> dict:
    y_true = df["y_true"].to_numpy()
    y_pred = df["y_pred"].to_numpy()

    acc = float(accuracy_score(y_true, y_pred))
    macro = float(f1_score(y_true, y_pred, labels=LABELS, average="macro", zero_division=0))

    f1_m1 = float(f1_score(y_true, y_pred, labels=[-1], average=None, zero_division=0)[0])
    f1_0  = float(f1_score(y_true, y_pred, labels=[0],  average=None, zero_division=0)[0])
    f1_p1 = float(f1_score(y_true, y_pred, labels=[1],  average=None, zero_division=0)[0])
    pm1_macro = (f1_m1 + f1_p1) / 2.0

    return {
        "test_accuracy": acc,
        "test_macro_f1": macro,
        "neutral_f1": f1_0,
        "pm1_macro_f1": pm1_macro,
        "f1_class_-1": f1_m1,
        "f1_class_0": f1_0,
        "f1_class_1": f1_p1,
    }

def mean_dict(dicts: list[dict]) -> dict:
    keys = dicts[0].keys()
    out = {}
    for k in keys:
        out[k] = float(np.mean([d[k] for d in dicts]))
    return out

def try_load_metric_row(exp_id: str) -> tuple[pd.Series | None, str]:
    p = METRICS_PATHS[exp_id]
    if not p.exists():
        return None, f"metrics missing: {p}"
    df = pd.read_csv(p)
    if "split" not in df.columns:
        return None, f"no split column: {p}"

    validation = META[exp_id][2]
    uniq_splits = sorted(pd.unique(df["split"].astype(str)))

    if validation == "V1":
        if "V1_test" in uniq_splits:
            cand = df[df["split"].astype(str) == "V1_test"]
            if cand.shape[0] == 1:
                return cand.iloc[0], f"matched split='V1_test' in {p}"
            return None, f"split='V1_test' not unique in {p}"
        if "test" in uniq_splits:
            cand = df[df["split"].astype(str) == "test"]
            if cand.shape[0] == 1:
                return cand.iloc[0], f"matched split='test' in {p}"
            return None, f"split='test' not unique in {p}"
        return None, f"no comparable V1 test split in metrics (splits={uniq_splits})"

    if validation == "V2":
        if "V2_test_mean" in uniq_splits:
            cand = df[df["split"].astype(str) == "V2_test_mean"]
            if cand.shape[0] == 1:
                return cand.iloc[0], f"matched split='V2_test_mean' in {p}"
            return None, f"split='V2_test_mean' not unique in {p}"
        return None, f"no comparable V2 aggregate row in metrics (splits={uniq_splits})"

    return None, "unknown validation"

def sanity_check(exp_id: str, computed: dict):
    row, reason = try_load_metric_row(exp_id)
    if row is None:
        print(f"[CHECK-SKIP] {exp_id}: {reason}")
        return

    for c in ["accuracy", "macro_f1"]:
        if c not in row.index:
            print("Not provable from provided evidence.")
            print(f"{exp_id}: comparable metrics row missing '{c}'. Row columns: {list(row.index)}")
            raise RuntimeError("STOP: cannot sanity-check")

    acc_m = float(row["accuracy"])
    f1_m  = float(row["macro_f1"])

    tol = 1e-12
    acc_ok = abs(acc_m - computed["test_accuracy"]) <= tol
    f1_ok  = abs(f1_m  - computed["test_macro_f1"]) <= tol

    if not (acc_ok and f1_ok):
        print("Not provable from provided evidence.")
        print(f"[CHECK-FAIL] {exp_id}: metrics vs predictions mismatch ({reason})")
        print(f" metrics accuracy={acc_m:.15f} | computed={computed['test_accuracy']:.15f}")
        print(f" metrics macro_f1={f1_m:.15f} | computed={computed['test_macro_f1']:.15f}")
        raise RuntimeError("STOP: metrics/predictions inconsistency")
    print(f"[CHECK-PASS] {exp_id}: accuracy+macro_f1 match metrics ({reason})")

print("\n--- Computing test metrics from predictions (V2 = mean across folds; fixed label order [-1,0,1]) ---")
rows = []
for exp_id in ["E1","E2","E3","E4","E5","E6","E7","E8"]:
    label_type, model, validation = META[exp_id]
    sources = PRED_SOURCES[exp_id]

    dfs = [load_pred_file(p) for p in sources]

    if validation == "V1":
        if len(dfs) != 1:
            print("Not provable from provided evidence.")
            print(f"{exp_id}: expected exactly 1 V1 predictions file, got {len(dfs)}.")
            raise RuntimeError("STOP: unexpected V1 prediction sources")
        df_all = dfs[0]
        computed = compute_metrics_from_df(df_all)

    else:
        fold_metrics = [compute_metrics_from_df(df_fold) for df_fold in dfs]
        computed = mean_dict(fold_metrics)

    df_concat = pd.concat(dfs, ignore_index=True)
    date_min = str(df_concat["date"].min())
    date_max = str(df_concat["date"].max())
    uniq_pred_union = sorted(pd.unique(df_concat["y_pred"]).tolist())
    n_rows = int(df_concat.shape[0])

    print(f"{exp_id}: n={n_rows} | date {date_min} -> {date_max} | uniq y_pred={uniq_pred_union} | validation={validation}")

    sanity_check(exp_id, computed)

    rows.append({
        "experiment": exp_id,
        "label_type": label_type,
        "model": model,
        "validation": validation,
        "test_accuracy": computed["test_accuracy"],
        "test_macro_f1": computed["test_macro_f1"],
        "neutral_f1": computed["neutral_f1"],
        "pm1_macro_f1": computed["pm1_macro_f1"],
        # keep per-class F1s for later auditability
        "f1_class_-1": computed["f1_class_-1"],
        "f1_class_0": computed["f1_class_0"],
        "f1_class_1": computed["f1_class_1"],
        "n_test_rows": n_rows,
        "date_min": date_min,
        "date_max": date_max,
        "uniq_y_pred_union": str(uniq_pred_union),
        "pred_sources": str([str(p) for p in sources]),
        "v2_aggregation_rule": ("mean_over_folds" if validation == "V2" else "single_file"),
    })

summary = pd.DataFrame(rows)

order = ["E1","E2","E3","E4","E5","E6","E7","E8"]
summary["experiment"] = pd.Categorical(summary["experiment"], categories=order, ordered=True)
summary = summary.sort_values("experiment").reset_index(drop=True)

print(summary[["experiment","label_type","model","validation","test_accuracy","test_macro_f1","neutral_f1","pm1_macro_f1"]].to_string(index=False))

summary.to_csv(OUT, index=False)
print(f"\n[WRITE] summary saved: {OUT}")


=== Week 8 — W8.8.2 (FIXED) Summary table from predictions (V2 = mean over folds) ===

--- Computing test metrics from predictions (V2 = mean across folds; fixed label order [-1,0,1]) ---
E1: n=1535 | date 2020-01-01 00:00:00 -> 2025-11-18 00:00:00 | uniq y_pred=[0] | validation=V1
[CHECK-PASS] E1: accuracy+macro_f1 match metrics (matched split='test' in ..\results\tables\metrics_E1.csv)
E2: n=2608 | date 2010-01-01 00:00:00 -> 2019-12-31 00:00:00 | uniq y_pred=[0, 1] | validation=V2
[CHECK-SKIP] E2: no comparable V2 aggregate row in metrics (splits=['test', 'train'])
E3: n=1535 | date 2020-01-01 00:00:00 -> 2025-11-18 00:00:00 | uniq y_pred=[0] | validation=V1
[CHECK-PASS] E3: accuracy+macro_f1 match metrics (matched split='test' in ..\results\tables\metrics_E3.csv)
E4: n=2608 | date 2010-01-01 00:00:00 -> 2019-12-31 00:00:00 | uniq y_pred=[0, 1] | validation=V2
[CHECK-SKIP] E4: no comparable V2 aggregate row in metrics (splits=['test', 'train'])
E5: n=1540 | date 2020-01-01 00:00:00 

In [ ]:
# W8.8.3 — Confusion matrices export (predictions-only) 

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 200)


TABLES = Path("../results/tables")
FIGS   = Path("../results/figures")
PREDS  = Path("../results/predictions")

if not TABLES.exists() or not PREDS.exists():
    print("Not provable from provided evidence.")
    print(f"Missing required directory. TABLES exists={TABLES.exists()} | PREDS exists={PREDS.exists()}")
    raise RuntimeError("STOP: missing required directory")

if not FIGS.exists():
    FIGS.mkdir(parents=True, exist_ok=True)
    print(f"[WRITE] created figures dir: {FIGS}")

LABELS = [-1, 0, 1]

def load_pred(p: Path) -> pd.DataFrame:
    if not p.exists():
        print("Not provable from provided evidence.")
        print(f"Missing predictions file: {p}")
        raise RuntimeError("STOP: missing predictions file")

    df = pd.read_csv(p)
    req = {"date", "y_true", "y_pred"}
    if not req.issubset(df.columns):
        print("Not provable from provided evidence.")
        print(f"{p} missing required columns {sorted(req)}. Found: {list(df.columns)}")
        raise RuntimeError("STOP: malformed predictions CSV")

    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    if df["date"].isna().any():
        bad = int(df["date"].isna().sum())
        print("Not provable from provided evidence.")
        print(f"{p} has {bad} rows where 'date' failed datetime parsing.")
        raise RuntimeError("STOP: bad date parsing")

    for col in ["y_true", "y_pred"]:
        x = df[col].dropna().to_numpy()
        if x.size == 0:
            print("Not provable from provided evidence.")
            print(f"{p}: column '{col}' is empty after dropping NaNs.")
            raise RuntimeError("STOP: empty label column")

        if np.issubdtype(x.dtype, np.floating):
            if not np.all(np.isclose(x, np.round(x))):
                print("Not provable from provided evidence.")
                print(f"{p}: column '{col}' contains non-integer float labels.")
                print("Distinct values (sample):", sorted(pd.unique(df[col].dropna()))[:20])
                raise RuntimeError("STOP: invalid label values")
            df[col] = np.round(df[col]).astype(int)
        else:
            df[col] = df[col].astype(int)

        uniq = sorted(pd.unique(df[col]).tolist())
        if not set(uniq).issubset(set(LABELS)):
            print("Not provable from provided evidence.")
            print(f"{p}: column '{col}' has labels outside {LABELS}: {uniq}")
            raise RuntimeError("STOP: label-space mismatch")

    return df

def confmat_df(y_true: np.ndarray, y_pred: np.ndarray) -> pd.DataFrame:
    cm = pd.DataFrame(
        0,
        index=[f"true_{l}" for l in LABELS],
        columns=[f"pred_{l}" for l in LABELS],
        dtype=int,
    )
    for t, p in zip(y_true, y_pred):
        cm.loc[f"true_{t}", f"pred_{p}"] += 1
    return cm

def save_cm(cm: pd.DataFrame, tag: str):
    out_csv = TABLES / f"confmat_{tag}.csv"
    out_png = FIGS   / f"confmat_{tag}.png"

    cm.to_csv(out_csv, index=True)

    fig, ax = plt.subplots(figsize=(5.4, 4.4))
    im = ax.imshow(cm.values)

    ax.set_xticks(range(len(LABELS)))
    ax.set_yticks(range(len(LABELS)))
    ax.set_xticklabels([str(l) for l in LABELS])
    ax.set_yticklabels([str(l) for l in LABELS])
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(f"Confusion Matrix — {tag} (raw counts)")

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, int(cm.values[i, j]), ha="center", va="center")

    fig.tight_layout()
    fig.savefig(out_png, dpi=200)
    plt.close(fig)

    print(f"[WRITE] {out_csv}")
    print(f"[WRITE] {out_png}")

# Targets per spec
targets_v1 = {
    "E5_V1_test": PREDS / "predictions_E5_V1_test.csv",
    "E6_V1_test": PREDS / "predictions_E6_V1_test.csv",
}

print("\n--- V1 confusion matrices (E5, E6) ---")
for tag, path in targets_v1.items():
    df = load_pred(path)
    cm = confmat_df(df["y_true"].values, df["y_pred"].values)
    print(f"\nTEXT MIRROR {tag} (rows={df.shape[0]} | date {df['date'].min()} -> {df['date'].max()}):")
    print(cm.to_string())
    save_cm(cm, tag)

print("\n--- V2 confusion matrices per fold + aggregated (E7, E8) ---")
for exp in ["E7", "E8"]:
    all_true = []
    all_pred = []

    for fold in ["F1", "F2", "F3", "F4"]:
        tag = f"{exp}_V2_{fold}_test"
        path = PREDS / f"predictions_{exp}_V2_{fold}_test.csv"
        df = load_pred(path)

        cm = confmat_df(df["y_true"].values, df["y_pred"].values)
        print(f"\nTEXT MIRROR {tag} (rows={df.shape[0]} | date {df['date'].min()} -> {df['date'].max()}):")
        print(cm.to_string())
        save_cm(cm, tag)

        all_true.append(df["y_true"].values)
        all_pred.append(df["y_pred"].values)

    all_true = np.concatenate(all_true)
    all_pred = np.concatenate(all_pred)
    cm_all = confmat_df(all_true, all_pred)
    tag_all = f"{exp}_V2_allfolds_test"
    print(f"\nTEXT MIRROR {tag_all} (rows={all_true.shape[0]} | aggregated folds F1..F4):")
    print(cm_all.to_string())
    save_cm(cm_all, tag_all)



=== Week 8 — W8.8.3 Confusion matrices export (predictions-only) ===

--- V1 confusion matrices (E5, E6) ---

TEXT MIRROR E5_V1_test (rows=1540 | date 2020-01-01 00:00:00 -> 2025-11-25 00:00:00):
         pred_-1  pred_0  pred_1
true_-1      194       0     263
true_0        33       0     235
true_1        43       0     772
[WRITE] ..\results\tables\confmat_E5_V1_test.csv
[WRITE] ..\results\figures\confmat_E5_V1_test.png

TEXT MIRROR E6_V1_test (rows=1540 | date 2020-01-01 00:00:00 -> 2025-11-25 00:00:00):
         pred_-1  pred_0  pred_1
true_-1      164       0     293
true_0        14       0     254
true_1        29       0     786
[WRITE] ..\results\tables\confmat_E6_V1_test.csv
[WRITE] ..\results\figures\confmat_E6_V1_test.png

--- V2 confusion matrices per fold + aggregated (E7, E8) ---

TEXT MIRROR E7_V2_F1_test (rows=521 | date 2010-01-01 00:00:00 -> 2011-12-30 00:00:00):
         pred_-1  pred_0  pred_1
true_-1       90       0      36
true_0        27       0      67
true_

In [ ]:
# W8.8.4 — Summary comparison plot (from summary table) 

from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 200)


TABLES = Path("../results/tables")
FIGS   = Path("../results/figures")

summary_path = TABLES / "summary_E5_E8_vs_E1_E4.csv"
out_png = FIGS / "summary_macroF1_E1_E8.png"

if not TABLES.exists() or not FIGS.exists():
    print("Not provable from provided evidence.")
    print(f"Missing directory. TABLES exists={TABLES.exists()} | FIGS exists={FIGS.exists()}")
    raise RuntimeError("STOP: missing required directory")

if not summary_path.exists():
    print("Not provable from provided evidence.")
    print(f"Missing summary table: {summary_path}")
    raise RuntimeError("STOP: missing summary table")

df = pd.read_csv(summary_path)

required_cols = {"experiment", "label_type", "validation", "test_macro_f1"}
if not required_cols.issubset(df.columns):
    print("Not provable from provided evidence.")
    print(f"Summary table missing required columns: {sorted(required_cols)}")
    print("Found:", list(df.columns))
    raise RuntimeError("STOP: malformed summary table")

# Deterministic experiment ordering
order = ["E1", "E2", "E3", "E4", "E5", "E6", "E7", "E8"]
df["experiment"] = pd.Categorical(df["experiment"].astype(str), categories=order, ordered=True)
df = df.sort_values("experiment").reset_index(drop=True)

df["test_macro_f1"] = pd.to_numeric(df["test_macro_f1"], errors="coerce")
if df["test_macro_f1"].isna().any():
    bad = int(df["test_macro_f1"].isna().sum())
    print("Not provable from provided evidence.")
    print(f"Found {bad} NaNs in test_macro_f1 after numeric coercion.")
    raise RuntimeError("STOP: invalid test_macro_f1 values")

print(df[["experiment", "label_type", "validation", "test_macro_f1"]].to_string(index=False))

fig, ax = plt.subplots(figsize=(8.5, 4.2))

xpos = {exp: i for i, exp in enumerate(order)}

for label_type in ["L1_Q", "L2"]:
    sub = df[df["label_type"] == label_type]
    for validation, marker in [("V1", "o"), ("V2", "s")]:
        ss = sub[sub["validation"] == validation]
        if ss.empty:
            continue
        ax.scatter(
            [xpos[str(e)] for e in ss["experiment"].astype(str)],
            ss["test_macro_f1"].astype(float),
            marker=marker,
            label=f"{label_type} / {validation}",
        )

ax.set_xticks(list(range(len(order))))
ax.set_xticklabels(order)
ax.set_xlabel("Experiment ID")
ax.set_ylabel("Test macro-F1")
ax.set_title("Test macro-F1 across E1–E8 (L1_Q vs L2; V1 vs V2)")
ax.legend(loc="best", frameon=True)

fig.tight_layout()
fig.savefig(out_png, dpi=200)
plt.close(fig)




=== Week 8 — W8.8.4 Summary macro-F1 plot (test) ===

=== TEXT MIRROR: plot data (experiment, label_type, validation, test_macro_f1) ===
experiment label_type validation  test_macro_f1
        E1       L1_Q         V1       0.281665
        E2       L1_Q         V2       0.238155
        E3       L1_Q         V1       0.281665
        E4       L1_Q         V2       0.238412
        E5         L2         V1       0.424743
        E6         L2         V1       0.408606
        E7         L2         V2       0.437053
        E8         L2         V2       0.431617

[WRITE] plot saved: ..\results\figures\summary_macroF1_E1_E8.png

=== COPY OUTPUT BACK ===


In [ ]:
# W8.8.5 — RQ-aligned factual notes (artifact-based, text-only) 
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 200)


TABLES = Path("../results/tables")
FIGS   = Path("../results/figures")
DOCS   = Path("../docs")

summary_path = TABLES / "summary_E5_E8_vs_E1_E4.csv"
required_confmats = {
    "E5_V1_test": TABLES / "confmat_E5_V1_test.csv",
    "E6_V1_test": TABLES / "confmat_E6_V1_test.csv",
    "E7_V2_allfolds_test": TABLES / "confmat_E7_V2_allfolds_test.csv",
    "E8_V2_allfolds_test": TABLES / "confmat_E8_V2_allfolds_test.csv",
}

# Existence gates
missing = []
if not summary_path.exists():
    missing.append(str(summary_path))
for k, p in required_confmats.items():
    if not p.exists():
        missing.append(str(p))

if missing:
    print("Not provable from provided evidence.")
    print("Missing required artifact(s) for W8.8.5:")
    for m in missing:
        print(" -", m)
    raise RuntimeError("STOP: missing required artifacts")

summary = pd.read_csv(summary_path)

req_cols = {"experiment", "label_type", "model", "validation", "test_accuracy", "test_macro_f1", "neutral_f1", "pm1_macro_f1"}
if not req_cols.issubset(summary.columns):
    print("Not provable from provided evidence.")
    print(f"Summary table missing required columns: {sorted(req_cols)}")
    print("Found:", list(summary.columns))
    raise RuntimeError("STOP: malformed summary table")

def get_row(exp_id: str) -> pd.Series:
    r = summary[summary["experiment"].astype(str) == exp_id]
    if r.shape[0] != 1:
        print("Not provable from provided evidence.")
        print(f"Expected exactly 1 row for experiment={exp_id}, found {r.shape[0]}.")
        raise RuntimeError("STOP: ambiguous summary indexing")
    return r.iloc[0]

LABELS = [-1, 0, 1]
def load_confmat(path: Path) -> pd.DataFrame:
    cm = pd.read_csv(path, index_col=0)
    # Required fixed-order schema from W8.8.3
    expected_cols = [f"pred_{l}" for l in LABELS]
    expected_rows = [f"true_{l}" for l in LABELS]
    if list(cm.columns) != expected_cols or list(cm.index) != expected_rows:
        print("Not provable from provided evidence.")
        print(f"Confmat schema mismatch in {path}")
        print("Expected cols:", expected_cols, "Found:", list(cm.columns))
        print("Expected idx :", expected_rows, "Found:", list(cm.index))
        raise RuntimeError("STOP: malformed confusion matrix schema")
    return cm

def pred_class_counts(cm: pd.DataFrame) -> dict:
    return {c: int(cm[c].sum()) for c in cm.columns}

def true_class_counts(cm: pd.DataFrame) -> dict:
    return {r: int(cm.loc[r].sum()) for r in cm.index}

def total_n(cm: pd.DataFrame) -> int:
    return int(cm.values.sum())

# Load needed rows
E1 = get_row("E1"); E2 = get_row("E2"); E3 = get_row("E3"); E4 = get_row("E4")
E5 = get_row("E5"); E6 = get_row("E6"); E7 = get_row("E7"); E8 = get_row("E8")

# Load confusion matrices (for concrete, file-backed counts)
cm_E5 = load_confmat(required_confmats["E5_V1_test"])
cm_E6 = load_confmat(required_confmats["E6_V1_test"])
cm_E7 = load_confmat(required_confmats["E7_V2_allfolds_test"])
cm_E8 = load_confmat(required_confmats["E8_V2_allfolds_test"])

pred_counts_E5 = pred_class_counts(cm_E5)
pred_counts_E6 = pred_class_counts(cm_E6)
pred_counts_E7 = pred_class_counts(cm_E7)
pred_counts_E8 = pred_class_counts(cm_E8)

true_counts_E5 = true_class_counts(cm_E5)
true_counts_E6 = true_class_counts(cm_E6)
true_counts_E7 = true_class_counts(cm_E7)
true_counts_E8 = true_class_counts(cm_E8)

n_E5 = total_n(cm_E5)
n_E6 = total_n(cm_E6)
n_E7 = total_n(cm_E7)
n_E8 = total_n(cm_E8)

# Compose factual notes (no interpretation beyond file-backed comparisons)
out_md = DOCS / "week08_diagnostic_findings.md"
DOCS.mkdir(parents=True, exist_ok=True)

lines = []
lines.append("# Week 8 Diagnostic Findings (Artifact-Based)")
lines.append("")
lines.append("## RQ2 (Label design: L1_Q vs L2)")
lines.append(f"- `results/tables/summary_E5_E8_vs_E1_E4.csv`: L1_Q baseline test macro-F1 values (artifact-derived) are "
             f"E1={float(E1['test_macro_f1']):.6f}, E2={float(E2['test_macro_f1']):.6f}, "
             f"E3={float(E3['test_macro_f1']):.6f}, E4={float(E4['test_macro_f1']):.6f}.")
lines.append(f"- `results/tables/summary_E5_E8_vs_E1_E4.csv`: L2 core test macro-F1 values (artifact-derived) are "
             f"E5={float(E5['test_macro_f1']):.6f}, E6={float(E6['test_macro_f1']):.6f}, "
             f"E7={float(E7['test_macro_f1']):.6f}, E8={float(E8['test_macro_f1']):.6f}.")
lines.append(f"- `results/tables/summary_E5_E8_vs_E1_E4.csv`: Neutral-class F1 (class 0) for L2 experiments is "
             f"E5={float(E5['neutral_f1']):.6f}, E6={float(E6['neutral_f1']):.6f}, "
             f"E7={float(E7['neutral_f1']):.6f}, E8={float(E8['neutral_f1']):.6f}.")
lines.append(f"- `results/tables/summary_E5_E8_vs_E1_E4.csv`: ±1 macro-F1 (mean of class -1 and +1 F1) for L2 experiments is "
             f"E5={float(E5['pm1_macro_f1']):.6f}, E6={float(E6['pm1_macro_f1']):.6f}, "
             f"E7={float(E7['pm1_macro_f1']):.6f}, E8={float(E8['pm1_macro_f1']):.6f}.")
lines.append(f"- `results/tables/confmat_E5_V1_test.csv`: predicted class counts on E5 V1 test (n={n_E5}) are "
             f"{pred_counts_E5} and true class counts are {true_counts_E5}.")
lines.append(f"- `results/tables/confmat_E6_V1_test.csv`: predicted class counts on E6 V1 test (n={n_E6}) are "
             f"{pred_counts_E6} and true class counts are {true_counts_E6}.")
lines.append(f"- `results/tables/confmat_E7_V2_allfolds_test.csv`: predicted class counts on E7 V2 all-folds test (n={n_E7}) are "
             f"{pred_counts_E7} and true class counts are {true_counts_E7}.")
lines.append(f"- `results/tables/confmat_E8_V2_allfolds_test.csv`: predicted class counts on E8 V2 all-folds test (n={n_E8}) are "
             f"{pred_counts_E8} and true class counts are {true_counts_E8}.")
lines.append("")
lines.append("## RQ3 (Simple models vs baselines)")
lines.append(f"- `results/tables/summary_E5_E8_vs_E1_E4.csv`: Under V1, baseline L1_Q Logistic vs Linear SVM test macro-F1 are "
             f"E1={float(E1['test_macro_f1']):.6f} vs E3={float(E3['test_macro_f1']):.6f}.")
lines.append(f"- `results/tables/summary_E5_E8_vs_E1_E4.csv`: Under V1, core L2 Logistic vs Linear SVM test macro-F1 are "
             f"E5={float(E5['test_macro_f1']):.6f} vs E6={float(E6['test_macro_f1']):.6f}.")
lines.append(f"- `results/tables/summary_E5_E8_vs_E1_E4.csv`: Under V2, baseline L1_Q Logistic vs Linear SVM test macro-F1 are "
             f"E2={float(E2['test_macro_f1']):.6f} vs E4={float(E4['test_macro_f1']):.6f}.")
lines.append(f"- `results/tables/summary_E5_E8_vs_E1_E4.csv`: Under V2, core L2 Logistic vs Linear SVM test macro-F1 are "
             f"E7={float(E7['test_macro_f1']):.6f} vs E8={float(E8['test_macro_f1']):.6f}.")
lines.append("")
lines.append("## Figures generated in W8.8")
lines.append("- `results/figures/summary_macroF1_E1_E8.png` (macro-F1 comparison plot from summary table).")
lines.append("- Confusion-matrix PNGs in `results/figures/confmat_*.png` (raw counts; fixed label order [-1,0,1]).")

out_md.write_text("\n".join(lines) + "\n", encoding="utf-8")

print(f"[WRITE] diagnostic notes saved: {out_md}")




=== Week 8 — W8.8.5 RQ-aligned factual notes (artifact-based) ===
[WRITE] diagnostic notes saved: ..\docs\week08_diagnostic_findings.md

=== TEXT MIRROR: first 25 lines of the written markdown ===
# Week 8 Diagnostic Findings (Artifact-Based)

## RQ2 (Label design: L1_Q vs L2)
- `results/tables/summary_E5_E8_vs_E1_E4.csv`: L1_Q baseline test macro-F1 values (artifact-derived) are E1=0.281665, E2=0.238155, E3=0.281665, E4=0.238412.
- `results/tables/summary_E5_E8_vs_E1_E4.csv`: L2 core test macro-F1 values (artifact-derived) are E5=0.424743, E6=0.408606, E7=0.437053, E8=0.431617.
- `results/tables/summary_E5_E8_vs_E1_E4.csv`: Neutral-class F1 (class 0) for L2 experiments is E5=0.000000, E6=0.000000, E7=0.000000, E8=0.000000.
- `results/tables/summary_E5_E8_vs_E1_E4.csv`: ±1 macro-F1 (mean of class -1 and +1 F1) for L2 experiments is E5=0.637114, E6=0.612910, E7=0.655579, E8=0.647425.
- `results/tables/confmat_E5_V1_test.csv`: predicted class counts on E5 V1 test (n=1540) are {'pred_-1':

In [ ]:
# W8.8.6 (PATCH) — Same log entry, but avoid datetime.utcnow() deprecation warning


from pathlib import Path
import datetime
import sklearn


DOCS = Path("../docs")
DOCS.mkdir(parents=True, exist_ok=True)
log_path = DOCS / "reproducibility_notes.md"

# FIX: timezone-aware UTC timestamp (avoids utcnow() deprecation)
ts = datetime.datetime.now(datetime.UTC).strftime("%Y-%m-%d %H:%M %Z")
skv = sklearn.__version__

entry = []
entry.append("")
entry.append(f"## Week 8 — Diagnostics consolidation (W8.8) — {ts}")

text_to_append = "\n".join(entry) + "\n"

if log_path.exists():
    existing = log_path.read_text(encoding="utf-8")
    log_path.write_text(existing + text_to_append, encoding="utf-8")
else:
    header = "# Reproducibility Notes\n"
    log_path.write_text(header + text_to_append, encoding="utf-8")




=== Week 8 — W8.8.6 Reproducibility log entry (PATCH: tz-aware UTC timestamp) ===
[WRITE] reproducibility notes updated: ..\docs\reproducibility_notes.md

=== TEXT MIRROR: appended entry ===

## Week 8 — Diagnostics consolidation (W8.8) — 2025-12-13 21:51 UTC
- Environment: scikit-learn==1.7.2.
- Observed warning: `multi_class` deprecation warning from scikit-learn during LogisticRegression usage (treated as version-sensitivity risk, not an error).
- Why this does not invalidate Week-8 results: all Week-8 metrics and prediction artifacts were generated and saved under the recorded environment; the warning indicates future API/default behavior changes, not incorrect execution in this run.
- Version-sensitivity risk: scikit-learn states `multi_class` will be removed; future versions may enforce multinomial behavior. Re-running scripts under a newer version without pinning constructors/defaults could change model behavior and thus metrics/predictions.
- Mitigation: keep environment pins (

In [ ]:
# W8.8.7 — Final deliverables audit  //// no need 

from pathlib import Path
import pandas as pd


BASE = Path("../")
required = [
    BASE / "results/tables/summary_E5_E8_vs_E1_E4.csv",

    BASE / "results/tables/confmat_E5_V1_test.csv",
    BASE / "results/tables/confmat_E6_V1_test.csv",
    BASE / "results/tables/confmat_E7_V2_F1_test.csv",
    BASE / "results/tables/confmat_E7_V2_F2_test.csv",
    BASE / "results/tables/confmat_E7_V2_F3_test.csv",
    BASE / "results/tables/confmat_E7_V2_F4_test.csv",
    BASE / "results/tables/confmat_E7_V2_allfolds_test.csv",
    BASE / "results/tables/confmat_E8_V2_F1_test.csv",
    BASE / "results/tables/confmat_E8_V2_F2_test.csv",
    BASE / "results/tables/confmat_E8_V2_F3_test.csv",
    BASE / "results/tables/confmat_E8_V2_F4_test.csv",
    BASE / "results/tables/confmat_E8_V2_allfolds_test.csv",

    BASE / "results/figures/summary_macroF1_E1_E8.png",
    BASE / "docs/week08_diagnostic_findings.md",
    BASE / "docs/reproducibility_notes.md",
]

rows = []
for p in required:
    exists = p.exists()
    size = p.stat().st_size if exists else 0
    rows.append({"path": str(p), "exists": exists, "bytes": int(size)})

df = pd.DataFrame(rows)

print(df.to_string(index=False))

missing = df.loc[~df["exists"], "path"].tolist()
empty = df.loc[(df["exists"]) & (df["bytes"] == 0), "path"].tolist()

if missing:
    print("[FAIL] Missing files:")
    for m in missing:
        print(" -", m)
elif empty:
    print("[FAIL] Empty (0-byte) files:")
    for e in empty:
        print(" -", e)
else:
    print("[PASS] All required deliverables exist and are non-empty.")



=== Week 8 — W8.8.7 Final deliverables audit (existence + non-empty) ===
                                             path  exists  bytes
     ..\results\tables\summary_E5_E8_vs_E1_E4.csv    True   2939
         ..\results\tables\confmat_E5_V1_test.csv    True     77
         ..\results\tables\confmat_E6_V1_test.csv    True     77
      ..\results\tables\confmat_E7_V2_F1_test.csv    True     74
      ..\results\tables\confmat_E7_V2_F2_test.csv    True     75
      ..\results\tables\confmat_E7_V2_F3_test.csv    True     74
      ..\results\tables\confmat_E7_V2_F4_test.csv    True     75
..\results\tables\confmat_E7_V2_allfolds_test.csv    True     80
      ..\results\tables\confmat_E8_V2_F1_test.csv    True     74
      ..\results\tables\confmat_E8_V2_F2_test.csv    True     75
      ..\results\tables\confmat_E8_V2_F3_test.csv    True     74
      ..\results\tables\confmat_E8_V2_F4_test.csv    True     76
..\results\tables\confmat_E8_V2_allfolds_test.csv    True     80
     ..\results\f

In [ ]:
from __future__ import annotations

from pathlib import Path
import re
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

try:
    import yaml  
except Exception as e:
    raise ImportError(
        "PyYAML is required "
    ) from e

try:
    from sklearn.metrics import cohen_kappa_score
except Exception as e:
    raise ImportError(
        "scikit-learn "
    ) from e



def find_project_root(start: Path) -> Path:
    cur = start.resolve()
    for _ in range(12):
        if (cur / "config" / "labels_features.yaml").exists():
            return cur
        if cur.parent == cur:
            break
        cur = cur.parent
    raise FileNotFoundError("Could not locate project root ")


def read_yaml(path: Path) -> dict:
    with path.open("r", encoding="utf-8") as f:
        return yaml.safe_load(f)


def ensure_dir(p: Path) -> None:
    p.mkdir(parents=True, exist_ok=True)


def parse_dates(df: pd.DataFrame, col: str = "date") -> pd.DataFrame:
    if col not in df.columns:
        raise KeyError(f"Missing required column '{col}'")
    df = df.copy()
    df[col] = pd.to_datetime(df[col], errors="coerce")
    bad = int(df[col].isna().sum())
    if bad:
        raise ValueError(f"Found {bad} NaT values after parsing '{col}'")
    return df


def save_fig(fig: plt.Figure, out_png: Path, out_pdf: Path) -> None:
    ensure_dir(out_png.parent)
    fig.savefig(out_png, dpi=300, bbox_inches="tight")
    fig.savefig(out_pdf, bbox_inches="tight")
    plt.close(fig)


def _add_bar_labels(ax: plt.Axes) -> None:
    for container in ax.containers:
        ax.bar_label(container, padding=2, fontsize=8)


def plot_validation_timeline(
    out_dir: Path,
    data_min: pd.Timestamp,
    data_max: pd.Timestamp,
) -> None:

    # V1 boundaries 
    V1 = {
        "train": (pd.Timestamp("2000-01-03"), pd.Timestamp("2015-12-31")),
        "val":   (pd.Timestamp("2016-01-01"), pd.Timestamp("2019-12-31")),
        "test":  (pd.Timestamp("2020-01-01"), pd.Timestamp(data_max)),
    }

    # V2 folds 
    V2 = [
        ("F1", (pd.Timestamp("2000-01-03"), pd.Timestamp("2009-12-31")),
               (pd.Timestamp("2010-01-01"), pd.Timestamp("2011-12-31"))),
        ("F2", (pd.Timestamp("2000-01-03"), pd.Timestamp("2011-12-31")),
               (pd.Timestamp("2012-01-01"), pd.Timestamp("2013-12-31"))),
        ("F3", (pd.Timestamp("2000-01-03"), pd.Timestamp("2013-12-31")),
               (pd.Timestamp("2014-01-01"), pd.Timestamp("2015-12-31"))),
        ("F4", (pd.Timestamp("2000-01-03"), pd.Timestamp("2015-12-31")),
               (pd.Timestamp("2016-01-01"), pd.Timestamp("2019-12-31"))),
    ]

    # Quick sanity (non-fatal, but warns)
    if data_min > pd.Timestamp("2000-01-03"):
        warnings.warn(f"Data starts at {data_min.date()}, later than expected 2000-01-03.")
    if data_max < pd.Timestamp("2020-01-01"):
        warnings.warn(f"Data ends at {data_max.date()}, earlier than expected 2020+ coverage.")

    fig, ax = plt.subplots(figsize=(12, 3.8))

    def seg_to_num(a: pd.Timestamp, b: pd.Timestamp) -> tuple[float, float]:
        a_num = mdates.date2num(a)
        b_num = mdates.date2num(b)
        return a_num, (b_num - a_num)

    # Y positions
    y_v1 = 25
    h = 8
    y0 = 10  

    # V1 blocks
    ax.broken_barh([seg_to_num(*V1["train"])], (y_v1, h), label="V1 train")
    ax.broken_barh([seg_to_num(*V1["val"])],   (y_v1, h), label="V1 val")
    ax.broken_barh([seg_to_num(*V1["test"])],  (y_v1, h), label="V1 test")

    ax.text(mdates.date2num(V1["train"][0]), y_v1 + h + 1, "V1", fontsize=9, va="bottom")

    for i, (fold, tr, te) in enumerate(V2):
        y = y0 + i * 10
        ax.broken_barh([seg_to_num(*tr)], (y, h), label="V2 train" if i == 0 else None)
        ax.broken_barh([seg_to_num(*te)], (y, h), label="V2 test" if i == 0 else None)
        ax.text(mdates.date2num(data_min) - 200, y + h/2, fold, fontsize=8, va="center")

    ax.set_ylim(0, 45 + 10 * (len(V2)-1))
    ax.set_xlim(mdates.date2num(data_min), mdates.date2num(data_max))
    ax.set_yticks([])
    ax.xaxis.set_major_locator(mdates.YearLocator(5))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.set_title("Validation schemes used in Week 8 (V1 chronological holdout + V2 walk-forward folds)")
    ax.grid(True, axis="x", linestyle="--", linewidth=0.5)

    # Compact legend
    handles, labels = ax.get_legend_handles_labels()
    # de-duplicate labels
    seen = set()
    H, L = [], []
    for hndl, lab in zip(handles, labels):
        if lab not in seen:
            seen.add(lab)
            H.append(hndl)
            L.append(lab)
    ax.legend(H, L, loc="upper left", fontsize=8)

    out_png = out_dir / "validation_timeline_V1_V2.png"
    out_pdf = out_dir / "validation_timeline_V1_V2.pdf"
    save_fig(fig, out_png, out_pdf)
    print(f"[OK] Wrote {out_png} and {out_pdf}")


def plot_true_vs_pred_counts(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    title: str,
    out_dir: Path,
    out_stem: str,
    label_order: list[int] = [-1, 0, 1],
) -> None:
    true_counts = [int(np.sum(y_true == c)) for c in label_order]
    pred_counts = [int(np.sum(y_pred == c)) for c in label_order]

    x = np.arange(len(label_order))
    width = 0.38

    fig, ax = plt.subplots(figsize=(7.5, 4.2))
    ax.bar(x - width/2, true_counts, width, label="True")
    ax.bar(x + width/2, pred_counts, width, label="Predicted")

    ax.set_xticks(x)
    ax.set_xticklabels([str(c) for c in label_order])
    ax.set_ylabel("Count")
    ax.set_title(title)
    ax.legend()

    _add_bar_labels(ax)
    ax.grid(True, axis="y", linestyle="--", linewidth=0.5)

    out_png = out_dir / f"{out_stem}.png"
    out_pdf = out_dir / f"{out_stem}.pdf"
    save_fig(fig, out_png, out_pdf)
    print(f"[OK] Wrote {out_png} and {out_pdf}")


def build_pred_class_freq_figures(pred_dir: Path, fig_dir: Path) -> None:
    
    if not pred_dir.exists():
        raise FileNotFoundError(f"Missing predictions directory: {pred_dir}")

    pred_files = sorted(pred_dir.glob("predictions_*.csv"))
    if not pred_files:
        raise FileNotFoundError(f"No prediction files found in: {pred_dir}")

    # First: per-file plots
    for p in pred_files:
        df = pd.read_csv(p)
        required = {"y_true", "y_pred"}
        if not required.issubset(df.columns):
            warnings.warn(f"Skipping {p.name}: missing columns {required - set(df.columns)}")
            continue

        # Convert to ints safely (your files are -1/0/1)
        y_true = pd.to_numeric(df["y_true"], errors="coerce").dropna().astype(int).to_numpy()
        y_pred = pd.to_numeric(df["y_pred"], errors="coerce").dropna().astype(int).to_numpy()

        # Use filename-derived title
        title = f"True vs predicted class counts — {p.stem}"
        out_stem = f"pred_class_freq_{p.stem}"

        plot_true_vs_pred_counts(y_true, y_pred, title, fig_dir, out_stem)

    # Second: aggregate "allfolds" for experiments with V2 folds
    # Pattern: predictions_E7_V2_F1_test.csv -> experiment key = E7_V2
    rx = re.compile(r"^predictions_(E\d+)_V2_F(\d+)_test\.csv$", re.IGNORECASE)

    buckets: dict[str, list[Path]] = {}
    for p in pred_files:
        m = rx.match(p.name)
        if not m:
            continue
        exp = m.group(1).upper()
        fold = m.group(2)
        key = f"{exp}_V2"
        buckets.setdefault(key, []).append(p)

    for key, paths in buckets.items():
        # only aggregate if we have >=2 folds
        if len(paths) < 2:
            continue
        dfs = []
        for p in sorted(paths):
            df = pd.read_csv(p)
            if {"y_true", "y_pred"}.issubset(df.columns):
                dfs.append(df[["y_true", "y_pred"]].copy())

        if not dfs:
            continue

        all_df = pd.concat(dfs, ignore_index=True)
        y_true = pd.to_numeric(all_df["y_true"], errors="coerce").dropna().astype(int).to_numpy()
        y_pred = pd.to_numeric(all_df["y_pred"], errors="coerce").dropna().astype(int).to_numpy()

        title = f"True vs predicted class counts — predictions_{key}_allfolds_test"
        out_stem = f"pred_class_freq_predictions_{key}_allfolds_test"
        plot_true_vs_pred_counts(y_true, y_pred, title, fig_dir, out_stem)


def label_agreement_heatmap(
    l1_path: Path,
    l2_path: Path,
    l1_col: str,
    l2_col: str,
    out_fig_dir: Path,
    out_table_dir: Path,
) -> None:
    l1 = parse_dates(pd.read_csv(l1_path), "date")
    l2 = parse_dates(pd.read_csv(l2_path), "date")

    if l1_col not in l1.columns:
        raise KeyError(f"L1 label column '{l1_col}' not found in {l1_path.name}")
    if l2_col not in l2.columns:
        raise KeyError(f"L2 label column '{l2_col}' not found in {l2_path.name}")

    df = pd.merge(
        l1[["date", l1_col]],
        l2[["date", l2_col]],
        on="date",
        how="inner",
        validate="one_to_one",
    )

    df = df.dropna(subset=[l1_col, l2_col]).copy()
    df[l1_col] = pd.to_numeric(df[l1_col], errors="coerce").astype(int)
    df[l2_col] = pd.to_numeric(df[l2_col], errors="coerce").astype(int)

    label_order = [-1, 0, 1]
    ct = pd.crosstab(
        df[l1_col],
        df[l2_col],
        rownames=["L1_Q"],
        colnames=["L2"],
        dropna=False,
    ).reindex(index=label_order, columns=label_order, fill_value=0)

    # Cohen's kappa
    kappa = float(cohen_kappa_score(df[l1_col], df[l2_col], labels=label_order))

    # Save table
    ensure_dir(out_table_dir)
    out_csv = out_table_dir / "label_agreement_L1Q_vs_L2_counts.csv"
    ct.to_csv(out_csv)
    print(f"[OK] Wrote {out_csv}")

    # Plot heatmap
    fig, ax = plt.subplots(figsize=(6.2, 5.2))
    im = ax.imshow(ct.values)

    ax.set_xticks(np.arange(len(label_order)))
    ax.set_yticks(np.arange(len(label_order)))
    ax.set_xticklabels([str(x) for x in label_order])
    ax.set_yticklabels([str(x) for x in label_order])

    ax.set_xlabel("L2 label")
    ax.set_ylabel("L1_Q label")
    ax.set_title(f"Label agreement heatmap (L1_Q vs L2) — Cohen's κ = {kappa:.3f}")

    # annotate cells
    for i in range(ct.shape[0]):
        for j in range(ct.shape[1]):
            ax.text(j, i, int(ct.values[i, j]), ha="center", va="center", fontsize=9)

    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    out_png = out_fig_dir / "label_agreement_L1Q_vs_L2_heatmap.png"
    out_pdf = out_fig_dir / "label_agreement_L1Q_vs_L2_heatmap.pdf"
    save_fig(fig, out_png, out_pdf)
    print(f"[OK] Wrote {out_png} and {out_pdf}")



def main() -> None:
    root = find_project_root(Path.cwd())
    cfg = read_yaml(root / "config" / "labels_features.yaml")

    features_path = root / cfg["dataset"]["features_file"]
    l1_path = root / cfg["dataset"]["label_files"]["L1_Q"]
    l2_path = root / cfg["dataset"]["label_files"]["L2"]

    l1_col = cfg["labels"]["L1_Q"]["label_column"]
    l2_col = cfg["labels"]["L2"]["label_column"]

    pred_dir = root / "results" / "predictions"
    fig_dir = root / "results" / "figures"
    table_dir = root / "results" / "tables"

    ensure_dir(fig_dir)
    ensure_dir(table_dir)

    feat = parse_dates(pd.read_csv(features_path, usecols=["date"]), "date")
    data_min = feat["date"].min()
    data_max = feat["date"].max()

    plot_validation_timeline(fig_dir, data_min=data_min, data_max=data_max)

    build_pred_class_freq_figures(pred_dir, fig_dir)

    label_agreement_heatmap(
        l1_path=l1_path,
        l2_path=l2_path,
        l1_col=l1_col,
        l2_col=l2_col,
        out_fig_dir=fig_dir,
        out_table_dir=table_dir,
    )



if __name__ == "__main__":
    warnings.simplefilter("default")
    main()


[OK] Wrote C:\Users\simon\Desktop\my thesis\ThesisProject\results\figures\validation_timeline_V1_V2.png and C:\Users\simon\Desktop\my thesis\ThesisProject\results\figures\validation_timeline_V1_V2.pdf
[OK] Wrote C:\Users\simon\Desktop\my thesis\ThesisProject\results\figures\pred_class_freq_predictions_E1_V1_test.png and C:\Users\simon\Desktop\my thesis\ThesisProject\results\figures\pred_class_freq_predictions_E1_V1_test.pdf
[OK] Wrote C:\Users\simon\Desktop\my thesis\ThesisProject\results\figures\pred_class_freq_predictions_E2_V1_test.png and C:\Users\simon\Desktop\my thesis\ThesisProject\results\figures\pred_class_freq_predictions_E2_V1_test.pdf
[OK] Wrote C:\Users\simon\Desktop\my thesis\ThesisProject\results\figures\pred_class_freq_predictions_E2_V2_F1_test.png and C:\Users\simon\Desktop\my thesis\ThesisProject\results\figures\pred_class_freq_predictions_E2_V2_F1_test.pdf
[OK] Wrote C:\Users\simon\Desktop\my thesis\ThesisProject\results\figures\pred_class_freq_predictions_E2_V2_F2_t